# Axiom Data：开发者的存储与实现教程

从原始响应到发布和恢复，理解每一步保存什么、改什么，以及怎样独立运行。

## 1. 架构地图与最小入口

Axiom Data 将供应商的事实交给研究代码。先看一条数据如何走完这条路，再讨论接口和存储细节。

**Tushare → Raw → Canonical → Snapshot → Reader → Research**

| 层 | 解决的问题 | 保存什么、何时产生 | 改动的影响 |
|---|---|---|---|
| Raw | 以后能查到“供应商当时返回了什么” | 采集时保存响应字节、请求与真实接收记录；相同字节只存一份 | 补拉产生新的观察，原接收时间保留 |
| Canonical | 日期、单位和证券身份能一起使用 | 归一化时生成有类型的 Parquet，财务与事件保持原生经济键 | 映射错误要重建受影响域 |
| Snapshot | 一次研究始终读取同一组事实 | 发布时生成引用分区、合同、来源的 JSON 清单 | 新版本可以复用旧分区，旧清单继续存在 |
| Reader | 在某一时刻能读到哪些事实 | 查询时返回带时间和证券键的表、单位、来源与限制 | cutoff 或读取规则变化会改变可见集合 |
| Research | 把事实转为研究问题的答案 | 由研究者保存代码、参数、Feature、模型和结果 | 数据版本与研究版本各自固定 |

Snapshot 保存文件引用，更新一个月无需复制整库。`current` 是“最新已完成版本”的小指针；实验开始时将它解析成具体 Snapshot，后面沿用这个固定 ID。`view` 可以只是 Snapshot 加查询描述，日常读取直接执行并使用有界内存缓存；需要 Qlib 文件时才显式导出。

以下数字来自已保存的真实 Tushare 小样本。股票行情使用一年固定 1800 工程样本，财务与成员使用两证券的完整源样本，ETF 使用七基金的独立样本；这三份数据各有固定版本，范围在相应例子中说明。无需先下载十二年数据便可阅读输出。运行环境和数据路径在末尾附录解释。

In [1]:
from pathlib import Path
import sys
repo = Path.cwd() / 'axiom-docs'
if not repo.is_dir():
    repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/tutorial_support.py').exists())
sys.path.insert(0, str(repo / 'examples'))
from tutorial_support import *  # pinned local data; writes use TEMP


## 2. 一个真实 Raw 对象与接收记录

Raw 对象按响应内容哈希存一次，日志按每次观察追加请求和真实接收时间。两次相同响应可以引用同一对象；重复保存相同响应字节没有收益。下面只展示一条已采日线的接收元数据和精选字段，原全市场响应留在本地。对象摘要检查保证字节未变，不能证明供应商事实必然正确。

In [2]:
segment = MANIFEST['domains']['market_daily']
raw_id = segment['raw_batch_ids'][0]
receipt = DATA.store.get_raw(raw_id)
print('Raw receipt:', json.dumps({k:receipt.get(k) for k in
    ('batch_id','domain','observed_at','payload_sha256','payload_uri','status')},ensure_ascii=False,indent=2))
print('请求 endpoint:', receipt['request'].get('endpoint'))
print('日志:', 'raw/fetches.jsonl')

Raw receipt: {
  "batch_id": "b_4353503ecc7348929d16f6640c11b63d",
  "domain": "market_daily",
  "observed_at": "2026-09-28T13:19:12.500927+00:00",
  "payload_sha256": "28a1deda0d07ed87f2859c52612e01b43e29ebede86168dece88decff0c35acd",
  "payload_uri": "raw/objects/28a1deda0d07ed87f2859c52612e01b43e29ebede86168dece88decff0c35acd/payload.bin",
  "status": "success"
}
请求 endpoint: daily
日志: raw/fetches.jsonl


## 3. 从字段映射到 Parquet 分区

供应商响应通常是宽表字段与行。源 profile 声明原字段如何映射到稳定证券身份、交易日和规范字段，contract 声明类型与单位。股票成交量转换为 shares，金额按来源单位转换；ETF 单位由基金合同声明。Canonical 按域和月份保存、分区按内容寻址。一个字段变化无需再保存一整套 Raw；事实不变时分区可复用。下面用真正的 Arrow schema 和一行原数据核对转换。

In [3]:
from axiom_data.sources import _rows  # source inspection only
domain = MANIFEST['domains']['market_daily']
part = domain['partitions'][-1]
table = DATA.store.read_partition(part)
print('实际分区:', part['uri'])
print(table.schema)
display(table.slice(0, 3).to_pandas())
row = table.to_pylist()[0]
raw_record = DATA.store.get_raw(row['raw_batch_id'])
source_rows = _rows(DATA.store.read_raw_record(raw_record))
reverse = {v: k for k, v in IDS.items()}
source = next(r for r in source_rows if r['ts_code'] == reverse[row['security_id']]
              and r['trade_date'] == str(row['session']).replace('-', ''))
assert Decimal(str(source['vol']))*100 == row['volume_shares']
assert abs(float(Decimal(str(source['amount']))*1000)-row['amount_cny']) < 1e-6
assert source['close'] == row['close']
display(pd.DataFrame([{'field':'volume','source':source['vol'],'canonical':row['volume_shares'],'conversion':'手 × 100 → 股'},
 {'field':'amount','source':source['amount'],'canonical':row['amount_cny'],'conversion':'千元 × 1000 → 元'},
 {'field':'close','source':source['close'],'canonical':row['close'],'conversion':'CNY/share 不变'}]))
print('Raw保留全市场行数:', len(source_rows), '实际接收时间:', raw_record['observed_at'])


实际分区: canonical/market_daily/2026-08/fecaac38eb6d9407376f674d73493ddffc636a59307f22947b9ce6e5b237450a.parquet
security_id: string not null
session: date32[day] not null
revision_id: string not null
revision_sequence: int64
first_observed_at: timestamp[us, tz=UTC] not null
raw_batch_id: string not null
source_available_at: timestamp[us, tz=UTC]
evidence_ref: string
amount_cny: double
close: double
high: double
low: double
open: double
pre_close: double
volume_shares: int64


,security_id,session,revision_id,revision_sequence,first_observed_at,raw_batch_id,source_available_at,evidence_ref,amount_cny,close,high,low,open,pre_close,volume_shares
0,cnstock.600048.SH.20060731,2026-08-06,terminal:8e849278bc2fe026f2cef745652f7fe60c1bd...,NaN,2026-09-28 13:23:33.884401+00:00,b_c361ee29a3cf4673b70edb9301efb09a,NaT,None,1.000004e+09,5.27,5.30,5.08,5.12,5.15,193005734
1,cnstock.600179.SH.19981104,2026-08-10,terminal:e6723efef8056bc5b0f0c44be4cc7c2d3eb35...,NaN,2026-09-28 13:23:34.777815+00:00,b_bf660c96ec164fafa356cda684559aeb,NaT,None,1.000049e+08,4.24,4.28,4.22,4.25,4.24,23526200
2,cnstock.603283.SH.20171225,2026-08-05,terminal:c17bd135b484911e252ff7ad90a678c63a6de...,NaN,2026-09-28 13:23:33.048527+00:00,b_eefc33fd937944439a4ecd07c0e4c5c2,NaT,None,1.000171e+09,41.96,42.53,39.20,39.51,39.24,24282670


,field,source,canonical,conversion
0,volume,1930057.34,1.930057e+08,手 × 100 → 股
1,amount,1000004.30,1.000004e+09,千元 × 1000 → 元
2,close,5.27,5.270000e+00,CNY/share 不变


Raw保留全市场行数: 5533 实际接收时间: 2026-09-28T13:23:33.884401+00:00


## 4. Manifest、旧版本与 current 发布

Snapshot 内嵌各域合同、源映射、分区引用和 Raw 引用。新版本继承未变域，最后成功发布才切换 current；旧 JSON 与分区保持可读。下面打印真实引用和合同。Snapshot 本身是小清单，并未复制整库。

In [4]:
display(pd.DataFrame([{'domain': k, 'contract': v['contract']['contract_id'],
 'profile': v['source_profile']['id'], 'raw_refs': len(v['raw_batch_ids']),
 'partitions': len(v['partitions']), 'coverage_keys': list(v['coverage'])}
 for k, v in MANIFEST['domains'].items()]))
print(json.dumps({'snapshot_id': SNAPSHOT, 'parent_snapshot': MANIFEST['parent_snapshot'],
 'schema_version': MANIFEST['schema_version'], 'one_partition': part}, indent=2))
assert not (ROOT/'views').exists()
print('没有强制物化 View 目录。')


,domain,contract,profile,raw_refs,partitions,coverage_keys
0,adjustment_factors,local.adjustment_factors.terminal_observation.v1,tushare.local.adj_factor.v1,242,12,"[basis, observed_chunks]"
1,benchmark_daily,local.benchmark_daily.terminal_observation.v1,tushare.local.index_daily.v1,1,12,"[basis, observed_chunks]"
2,market_daily,local.market_daily.terminal_observation.v1,tushare.local.daily.v1,242,12,"[basis, observed_chunks]"
3,security_master,local.security_master.terminal_observation.v1,tushare.local.stock_basic.v1,6,1,"[basis, observed_chunks]"
4,security_status,local.security_status.terminal_observation.v1,tushare.local.suspend_d.v1,242,12,"[basis, observed_chunks]"
5,trading_calendar,local.trading_calendar.terminal_observation.v1,tushare.local.trade_cal.v1,2,12,"[basis, observed_chunks]"


{
  "snapshot_id": "s_c3faabc9ec124597e91701af33ebac563cd386eff776ac467f2a2462340bff2b",
  "parent_snapshot": "s_cc5468e5eba5f8cf05d0822db25d435d66db5711e1456f9239315ab783b25a5d",
  "schema_version": "local_data_v1",
  "one_partition": {
    "file_sha256": "fecaac38eb6d9407376f674d73493ddffc636a59307f22947b9ce6e5b237450a",
    "partition": "2026-08",
    "rows": 37789,
    "uri": "canonical/market_daily/2026-08/fecaac38eb6d9407376f674d73493ddffc636a59307f22947b9ce6e5b237450a.parquet"
  }
}
没有强制物化 View 目录。


## 5. Reader 如何应用时间与缺失规则

Reader 从固定版本读取所需列和分区，再按 policy/cutoff 选择可见事实。返回 DataBatch 的 frame、字段单位/来源和 context。逐字段来源元数据有成本，后续优化应保持语义等价；它不成为全量采集的新增门槛。[Researcher 时间例子](researcher_tutorial.html#section-4)解释历史假设与真实在线观察的区别。

同一财报的回采/及时日更/晚到/修订例子见 Researcher 第4节；其中文表格已由离线反例验证。Snapshot 覆盖一年，默认 DAYS 只选择末尾六个交易日。合法空值不阻止其他键读取；日内停牌归 `unknown_status`，具体原因在元数据。

| `market_state` | 中文含义与成立条件 | 研究如何使用 |
|---|---|---|
| `normal_trading` | 日历开市、身份有效、状态明确未全天停牌，所需行情字段均可见 | 继续使用所请求事实；不等于日内可成交证明 |
| `calendar_closed` | 可见日历明确闭市 | 诊断行保留，价格为空，不补成交易 bar |
| `not_listed` | 可见身份表明该日早于上市 | 价格为空，不属于该证券有效交易期 |
| `delisted` | 可见身份/生命周期表明已越过退市有效边界 | 价格为空；不推断来源未给的日内边界 |
| `suspended` | 可见来源明确全天停牌 | 保留停牌解释；行情不存在时保持空值 |
| `source_gap` | 身份/状态明确、声明范围覆盖，但所需行情事实缺失 | 保持空值及 `market_fact_missing`，定位对应源窗口 |
| `unknown_status` | 日历、身份、状态或行情可见性不足；也包括日内停牌 | 看 `market_state` 的字段元数据原因；不默认正常或全天停牌 |

日内停牌的具体原因是 `partial_session_suspension`；未到 cutoff 常见 `not_visible_at_cutoff`，没有源行常见 `source_missing`。这些合法缺失允许其他证券/日期继续读取。缺少所需域、未声明字段、单位/身份错误或文件损坏会明确报错，需要定位修正。`source_gap` 是固定 Snapshot 的覆盖诊断，不证明历史当天已经知道缺口。


In [5]:
batch = DATA.read(snapshot=SNAPSHOT, query=query(('open', 'close', 'volume_shares')))
display(batch.frame)
# Rows are time × security; fields are columns, not a new storage object per feature.
panel = DATA.read(snapshot=SNAPSHOT, query=query(('close',),
    symbols=tuple(IDS[c] for c in ('000001.SZ', '600000.SH', '600519.SH'))))
display(panel.frame.pivot(index='session', columns='security_id', values='close'))
print('单位:', {k: v['unit'] for k, v in batch.field_meta.items()})


,security_id,session,open,close,volume_shares
0,cnstock.000001.SZ.19910403,2026-08-24,11.38,11.56,119902495
1,cnstock.000001.SZ.19910403,2026-08-25,11.57,11.59,99488115
2,cnstock.000001.SZ.19910403,2026-08-26,11.55,11.73,116728878
3,cnstock.000001.SZ.19910403,2026-08-27,11.7,11.59,97570170
4,cnstock.000001.SZ.19910403,2026-08-28,11.54,11.65,83812624
5,cnstock.000001.SZ.19910403,2026-08-31,11.64,11.72,90885655


security_id,cnstock.000001.SZ.19910403,cnstock.600000.SH.19991110,cnstock.600519.SH.20010827
session,,,
2026-08-24,11.56,9.22,1304.66
2026-08-25,11.59,9.08,1304.0
2026-08-26,11.73,9.21,1302.8
2026-08-27,11.59,9.07,1292.3
2026-08-28,11.65,9.0,1297.4
2026-08-31,11.72,9.16,1299.52


单位: {'open': 'CNY/share', 'close': 'CNY/share', 'volume_shares': 'shares'}


In [6]:
comparison = []
for policy in ('operational_pit_v1', 'market_pit_safe_v1', 'best_effort_vendor_v1'):
    result = DATA.read(snapshot=SNAPSHOT, query=query(policy=policy))
    comparison.append({'policy': policy, 'rows': len(result.frame),
                       'visible_close': int(result.frame.close.notna().sum()),
                       'limitation': '; '.join(result.context['limitations'])})
display(pd.DataFrame(comparison))
assert comparison[0]['visible_close'] == comparison[1]['visible_close'] == 0
assert comparison[2]['visible_close'] == len(DAYS)
print('一条真实字段来源：')
display(pd.DataFrame(batch.field_meta['close']['by_key'][:1]))


,policy,rows,visible_close,limitation
0,operational_pit_v1,6,0,terminal states ordered by system observation ...
1,market_pit_safe_v1,6,0,6 queried revisions lacked revision-bound publ...
2,best_effort_vendor_v1,6,6,"vendor release time is an assumption, not hist..."


一条真实字段来源：


,security_id,session,revision_id,revision_sequence,raw_batch_id,usable_from,first_observed_at,availability_basis,evidence_ref,missing_reason
0,cnstock.000001.SZ.19910403,2026-08-24,terminal:221697a48826b852e85b280f032a9368a2282...,None,b_71ba3777f9674756bbd6805dcebf4dd4,2026-08-24T20:00:00+08:00,2026-09-28T13:23:41.124858+00:00,declared_vendor_assumption,None,None


In [7]:
missing = sorted(AUDIT['missing_market_samples'], key=lambda r: (r['security_id'], r['session']))[0]
closed_day = '2025-09-27'
state_query = query(symbols=(missing['security_id'],), sessions=(missing['session'], closed_day))
state_batch = DATA.states(snapshot=SNAPSHOT, query=state_query)
display(state_batch.frame)
assert set(state_batch.frame.market_state) == {'suspended', 'calendar_closed'}
master_rows = [r for p in MANIFEST['domains']['security_master']['partitions']
               for r in DATA.store.read_partition(p).to_pylist()]
new_listing = next(r for r in master_rows if str(r['listing_date']) > '2025-09-01')
past_knowledge = DATA.states(snapshot=SNAPSHOT,
    query=query(symbols=(new_listing['security_id'],), sessions=('2025-09-01',)))
display(past_knowledge.frame)
print('原日期 cutoff 尚不能看到后来上市身份；再用实际当前知识回顾该历史日期。')
pre_listing = DATA.states(snapshot=SNAPSHOT,
    query=query(symbols=(new_listing['security_id'],), sessions=('2025-09-01',),
                policy='operational_pit_v1', common_cutoff=datetime.now(timezone.utc).isoformat()))
display(pre_listing.frame)
assert pre_listing.frame.market_state.iloc[0] == 'not_listed'
print('全范围缺价:', AUDIT['missing_market_cells'],
      '有停牌证据:', AUDIT['known_suspension_missing_cells'], '未解释:',
      AUDIT['missing_market_cells'] - AUDIT['known_suspension_missing_cells'])


,security_id,session,market_state,close
0,cnstock.002185.SZ.20071120,2025-09-25,suspended,<NA>
1,cnstock.002185.SZ.20071120,2025-09-27,calendar_closed,None


,security_id,session,market_state,close
0,cnstock.603418.SH.20250923,2025-09-01,unknown_status,<NA>


原日期 cutoff 尚不能看到后来上市身份；再用实际当前知识回顾该历史日期。


,security_id,session,market_state,close
0,cnstock.603418.SH.20250923,2025-09-01,not_listed,None


全范围缺价: 158 有停牌证据: 158 未解释: 0


## 6. 完整来源的原生键与 Schema

行情以交易日和证券为键，财报保留 endpoint、report_type、报告期以及修订；股东以整组报告替换；分红包含过程状态。以下读取两证券完整源的真实 schema、单位与首行。空域保持空，资产存量不应用收入累计转单季公式。

统一字段入口为 `data.dictionary(snapshot=固定ID)`；代码支持与该 Snapshot 声明分开。只有显式传 `raw_batch_id` 才查看那一份已存响应的未映射列，不扫描全库 Raw。

In [8]:
# The unified Snapshot combines retained original Tushare Raw and dated vendor membership.
FULL_ROOT = UNIFIED_ROOT
full = UNIFIED
FULL_SNAPSHOT = UNIFIED_SNAPSHOT
full_manifest = full.store.load_snapshot(FULL_SNAPSHOT)
full_scope = json.loads((REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.scope.json').read_text())
bank_id = full_scope['identity_map']['600000.SH']
display(pd.DataFrame([{'domain':name,
    'rows':sum(p['rows'] for p in segment['partitions']),
    'raw_receipts':len(segment['raw_batch_ids'])}
    for name, segment in full_manifest['domains'].items()]))
fin_domains = ('financial_events', 'balance_sheet_events', 'cash_flow_events',
               'financial_indicator_events', 'corporate_actions',
               'top_holders_reports', 'price_limits')
assert all(name in full_manifest['domains'] for name in fin_domains), 'complete-source publication is required'

,domain,rows,raw_receipts
0,adjustment_factors,130,65
1,balance_sheet_events,13,20
2,cash_flow_events,10,20
3,corporate_actions,20,277
4,financial_events,14,20
5,financial_indicator_events,12,6
6,listing_events,5888,6
7,market_daily,130,65
8,price_limits,130,65
9,security_master,2,6


In [9]:
source_domains = fin_domains + ('listing_events', 'universe_membership')
assert all(name in full_manifest['domains'] for name in source_domains)
assert all(full_manifest['domains'][name]['source_profile']['id'].startswith('tushare.')
    for name in source_domains), 'production domains must use retained Tushare Raw'
inventory = []
for name in source_domains:
    segment = full_manifest['domains'][name]
    inventory.append({'domain':name, 'rows':sum(p['rows'] for p in segment['partitions']),
        'partitions':len(segment['partitions']), 'raw_receipts':len(segment['raw_batch_ids']),
        'source_profile':segment['source_profile']['id'],
        'native_key':', '.join(segment['contract']['logical_key'])})
display(pd.DataFrame(inventory))
print('统一完整源根:', FULL_ROOT, '固定版本:', FULL_SNAPSHOT)
market_ids = sorted({r['security_id'] for p in full_manifest['domains']['market_daily']['partitions']
    for r in full.store.read_partition(p).to_pylist()})

,domain,rows,partitions,raw_receipts,source_profile,native_key
0,financial_events,14,3,20,tushare.local.income.v1,"security_id, endpoint, report_type, report_period"
1,balance_sheet_events,13,3,20,tushare.local.balancesheet.v1,"security_id, endpoint, report_type, report_period"
2,cash_flow_events,10,3,20,tushare.local.cashflow.v1,"security_id, endpoint, report_type, report_period"
3,financial_indicator_events,12,2,6,tushare.local.fina_indicator.v1,"security_id, endpoint, report_period"
4,corporate_actions,20,3,277,tushare.local.dividend.v1,"security_id, report_period, announcement_date,..."
5,top_holders_reports,12,2,2,tushare.local.top10_holders.v1,"security_id, report_period"
6,price_limits,130,3,65,tushare.local.stk_limit.v1,"security_id, session"
7,listing_events,5888,1,6,tushare.stock_basic.listing_events.v1,"security_id, event_type"
8,universe_membership,3874,1,12,tushare.index_weight.dated_membership.v1,membership_id


统一完整源根: /Users/liuming/Documents/axiom/data/csi1800_tushare_only_v4_202606_202608 固定版本: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049


In [10]:
print('采集范围:', full_scope['start_session'], '至', full_scope['end_session'],
      '；市场证券:', ', '.join(market_ids), '；成员候选身份数:', len(full_scope['symbols']))
print('此 Snapshot 同时含 Tushare 成员观测；供应商源由留存原始 Raw 离线回放，无新在线请求。')

采集范围: 2026-06-01 至 2026-08-31 ；市场证券: cnstock.000001.SZ.19910403, cnstock.600000.SH.19991110 ；成员候选身份数: 1905
此 Snapshot 同时含 Tushare 成员观测；供应商源由留存原始 Raw 离线回放，无新在线请求。


In [11]:
facts = full.dictionary(snapshot=FULL_SNAPSHOT)
dictionary_frame = pd.DataFrame(facts['fields'])
available = dictionary_frame[dictionary_frame.snapshot_available.eq(True)]
display(available[['domain','raw_field','canonical_field','meaning','dtype','unit','basis','time_role','query_method']]
    .rename(columns={'domain':'域','raw_field':'Raw字段','canonical_field':'规范字段','meaning':'含义',
                     'dtype':'类型','unit':'规范单位','basis':'口径','time_role':'时间角色','query_method':'查询方法'}))
raw_id = full_manifest['domains']['financial_events']['raw_batch_ids'][0]
one_raw_dictionary = full.dictionary(snapshot=FULL_SNAPSHOT,
    domains=('financial_events',), raw_batch_id=raw_id)
raw_only = pd.DataFrame([r for r in one_raw_dictionary['fields'] if not r['supported']])
print('显式一份财务 Raw 中尚未映射字段数:', len(raw_only))
if not raw_only.empty: display(raw_only[['raw_field','status','query_method']].head(20))
for name in source_domains:
    segment = full_manifest['domains'][name]
    first = next((p for p in segment['partitions'] if p['rows']), None)
    if first is None:
        print(name, '有来源完成记录，但当前范围无规范事实行')
        continue
    table = full.store.read_partition(first)
    print(name, 'Arrow:', table.schema)
    display(table.slice(0, 1).to_pandas())

,域,Raw字段,规范字段,含义,类型,规范单位,口径,时间角色,查询方法
0,adjustment_factors,None,evidence_ref,修订公开时刻证据引用,string,None,None,None,Data.read / QuerySpec；security_id、session 为隐含键
1,adjustment_factors,adj_factor,factor,供应商累计复权因子；调整价使用因子比,float64,dimensionless,None,None,Data.read / QuerySpec；security_id、session 为隐含键
2,adjustment_factors,None,first_observed_at,本系统首次实际接收本修订的时刻,timestamp,None,None,系统知识时间,Data.read / QuerySpec；security_id、session 为隐含键
3,adjustment_factors,None,raw_batch_id,实际 Raw 接收记录引用,string,None,None,None,Data.read / QuerySpec；security_id、session 为隐含键
4,adjustment_factors,None,revision_id,内容修订身份,string,None,None,None,Data.read / QuerySpec；security_id、session 为隐含键
...,...,...,...,...,...,...,...,...,...
250,universe_membership,None,revision_id,内容修订身份,string,None,None,None,成员区间合同；查询使用 is_member 投影
251,universe_membership,None,revision_sequence,声明的修订次序,int64,None,None,None,成员区间合同；查询使用 is_member 投影
252,universe_membership,None,security_id,稳定证券身份,string,None,None,None,成员区间合同；查询使用 is_member 投影
253,universe_membership,None,source_available_at,修订绑定的市场公开时刻，须有证据,timestamp,None,None,有证据的市场知识时间,成员区间合同；查询使用 is_member 投影


显式一份财务 Raw 中尚未映射字段数: 0
financial_events Arrow: security_id: string not null
session: string
revision_id: string not null
revision_sequence: int64
first_observed_at: timestamp[us, tz=UTC] not null
raw_batch_id: string not null
source_available_at: timestamp[us, tz=UTC]
evidence_ref: string
endpoint: string not null
report_type: string not null
report_period: date32[day] not null
actual_announcement_date: date32[day] not null
announcement_date: date32[day] not null
basis: string not null
parent_net_income: double
total_revenue: double


,security_id,session,revision_id,revision_sequence,first_observed_at,raw_batch_id,source_available_at,evidence_ref,endpoint,report_type,report_period,actual_announcement_date,announcement_date,basis,parent_net_income,total_revenue
0,cnstock.000001.SZ.19910403,None,terminal:685cc395940ad7240bd9db12d1f5540adc89d...,NaN,2026-09-28 15:06:17.810743+00:00,b_f903f7b83f9a40ad9e8d9a9790e34a3a,NaT,None,income,1,2024-12-31,2025-03-15,2025-03-15,cumulative_ytd,4.450800e+10,1.466950e+11


balance_sheet_events Arrow: security_id: string not null
session: string
revision_id: string not null
revision_sequence: int64
first_observed_at: timestamp[us, tz=UTC] not null
raw_batch_id: string not null
source_available_at: timestamp[us, tz=UTC]
evidence_ref: string
endpoint: string not null
report_type: string not null
report_period: date32[day] not null
actual_announcement_date: date32[day] not null
announcement_date: date32[day] not null
parent_equity: double
total_assets: double
total_liabilities: double


,security_id,session,revision_id,revision_sequence,first_observed_at,raw_batch_id,source_available_at,evidence_ref,endpoint,report_type,report_period,actual_announcement_date,announcement_date,parent_equity,total_assets,total_liabilities
0,cnstock.000001.SZ.19910403,None,terminal:f77ce3f3129d2c0fe5356dd57be7863884062...,NaN,2026-09-28 15:06:23.107249+00:00,b_acb899db12fc4171a9992099c6bef8b2,NaT,None,balancesheet,1,2024-12-31,2025-03-15,2025-03-15,4.948420e+11,5.769270e+12,5.274428e+12


...

,security_id,session,revision_id,revision_sequence,first_observed_at,raw_batch_id,source_available_at,evidence_ref,down_limit,up_limit
0,cnstock.000001.SZ.19910403,2026-06-10,terminal:e2533cbf9450bc55df1047b1e3c8f02c9f3d3...,NaN,2026-10-03 03:08:17.095080+00:00,b_2d133e87d09e4cb4be23fd8a2f6a8c87,NaT,None,10.02,12.24


listing_events Arrow: security_id: string not null
session: string
revision_id: string not null
revision_sequence: int64 not null
first_observed_at: timestamp[us, tz=UTC] not null
raw_batch_id: string not null
source_available_at: timestamp[us, tz=UTC]
evidence_ref: string
event_type: string not null
delisting_date: date32[day]
event_date: date32[day] not null
event_state: string not null
exchange: string not null
listing_date: date32[day] not null
source_code: string not null
vendor_list_status: string not null


,security_id,session,revision_id,revision_sequence,first_observed_at,raw_batch_id,source_available_at,evidence_ref,event_type,delisting_date,event_date,event_state,exchange,listing_date,source_code,vendor_list_status
0,cnstock.000508.SZ.19930430,None,vendor:bb2544b540ff183572ff4d62ace66fa7ea5b5cd...,1,2026-09-28 13:16:38.388413+00:00,b_b98968e0974f4f5fb4e9a185aa41bc8e,NaT,None,delisting,1999-07-12,1999-07-12,value,SZSE,1993-04-30,000508.SZ,D


universe_membership Arrow: security_id: string not null
session: string
revision_id: string not null
revision_sequence: int64 not null
first_observed_at: timestamp[us, tz=UTC] not null
raw_batch_id: string not null
source_available_at: timestamp[us, tz=UTC]
evidence_ref: string
membership_id: string not null
effective_from: date32[day] not null
effective_to: date32[day]
universe_id: string not null
dependency_raw_batch_ids: list<element: string>
  child 0, element: string


,security_id,session,revision_id,revision_sequence,first_observed_at,raw_batch_id,source_available_at,evidence_ref,membership_id,effective_from,effective_to,universe_id,dependency_raw_batch_ids
0,cnstock.000661.SZ.19961218,None,vendor:534df288d9135700636a978dbf88c59a1a4cced...,1,2026-09-28 13:17:26.380734+00:00,b_1c3c1f6c19a948299796556564117c98,NaT,None,csi500:cnstock.000661.SZ.19961218:vendor:csi50...,2026-06-30,None,csi500,[b_1c3c1f6c19a948299796556564117c98]


In [12]:
finance_domain = full_manifest['domains']['financial_events']
finance_rows = [row for part in finance_domain['partitions']
                for row in full.store.read_partition(part, symbols=(bank_id,)).to_pylist()
                if row['security_id'] == bank_id and row['endpoint'] == 'income'
                and row['report_type'] == '1']
assert finance_rows, 'expected real Tushare consolidated income observations'
periods = sorted({str(row['report_period']) for row in finance_rows})
cutoff_now = datetime.now(timezone.utc).isoformat()
income_query = EventQuery('financial_events', ('total_revenue','parent_net_income'),
    (bank_id,), periods[0], periods[-1], cutoff_now, 'operational_pit_v1',
    'report_period', {'endpoint':'income','report_type':'1'})
income_events = full.events(snapshot=FULL_SNAPSHOT, query=income_query)
display(income_events.frame)
assert len(income_events.frame) > 0
assert income_events.field_meta['total_revenue']['unit'] == 'CNY'
display(pd.DataFrame(income_events.field_meta['total_revenue']['by_key']))
print('原生报告键:', finance_domain['contract']['logical_key'])
print('Reader 先按 cutoff 选可见 revision，再按报告期和 endpoint 投影。')


,security_id,endpoint,report_type,report_period,total_revenue,parent_net_income
0,cnstock.600000.SH.19991110,income,1,2024-12-31,170748000000.0,45257000000.0
1,cnstock.600000.SH.19991110,income,1,2025-03-31,45922000000.0,17598000000.0
2,cnstock.600000.SH.19991110,income,1,2025-06-30,90559000000.0,29737000000.0
3,cnstock.600000.SH.19991110,income,1,2025-09-30,132280000000.0,38819000000.0
4,cnstock.600000.SH.19991110,income,1,2025-12-31,173964000000.0,50017000000.0
5,cnstock.600000.SH.19991110,income,1,2026-03-31,46573000000.0,17861000000.0
6,cnstock.600000.SH.19991110,income,1,2026-06-30,93777000000.0,30951000000.0


,security_id,endpoint,report_type,report_period,status,missing_reason,revision_id,revision_sequence,raw_batch_id,usable_from,first_observed_at,availability_basis,evidence_ref,group_completeness
0,cnstock.600000.SH.19991110,income,1,2024-12-31,value,None,terminal:be290169fc86e93935a7cfd66d3215b01dc2a...,None,b_f903f7b83f9a40ad9e8d9a9790e34a3a,2026-09-28T15:06:17.810743+00:00,2026-09-28T15:06:17.810743+00:00,first_observed_at,None,None
1,cnstock.600000.SH.19991110,income,1,2025-03-31,value,None,terminal:74ad5117e58f679f11a4c2852a6ae0881587e...,None,b_a4945cfc079d459dbd7e06ad259defcd,2026-09-28T15:06:18.283748+00:00,2026-09-28T15:06:18.283748+00:00,first_observed_at,None,None
2,cnstock.600000.SH.19991110,income,1,2025-06-30,value,None,terminal:542c8508772d0cefbdfd7b82bd04bfaae49d2...,None,b_0a4c0b5c5deb48ef9525b756fe5571a6,2026-09-28T15:06:19.027179+00:00,2026-09-28T15:06:19.027179+00:00,first_observed_at,None,None
3,cnstock.600000.SH.19991110,income,1,2025-09-30,value,None,terminal:e22928a1a1240c4818855692246014d478767...,None,b_98ad9b3093324b14a3c7e604da35def7,2026-09-28T15:06:19.412095+00:00,2026-09-28T15:06:19.412095+00:00,first_observed_at,None,None
4,cnstock.600000.SH.19991110,income,1,2025-12-31,value,None,terminal:060cd8f2487503accb8b607a2bf4041ee7c0f...,None,b_1a7403b5c1064591ba46560a43e121d6,2026-09-28T15:06:20.459198+00:00,2026-09-28T15:06:20.459198+00:00,first_observed_at,None,None
5,cnstock.600000.SH.19991110,income,1,2026-03-31,value,None,terminal:623990f554cf16d06bcc6eaca605acb0ac272...,None,b_1cf2fbd0e3054f46a32c7db075736fa6,2026-09-28T15:06:21.037797+00:00,2026-09-28T15:06:21.037797+00:00,first_observed_at,None,None
6,cnstock.600000.SH.19991110,income,1,2026-06-30,value,None,terminal:3f7ec15b3818615c970c12e0668141c8d7811...,None,b_603ea7e454994dffbcb995a44c2d35bd,2026-09-28T15:06:22.091878+00:00,2026-09-28T15:06:22.091878+00:00,first_observed_at,None,None


原生报告键: ['security_id', 'endpoint', 'report_type', 'report_period']
Reader 先按 cutoff 选可见 revision，再按报告期和 endpoint 投影。


In [13]:
# Compare the same native report keys at a historical research clock.
historical_clock = '2026-06-01T09:30:00+08:00'
pit_rows = []
for policy in ('operational_pit_v1', 'market_pit_safe_v1', 'best_effort_vendor_v1'):
    selected = full.events(snapshot=FULL_SNAPSHOT,
        query=replace(income_query, cutoff=historical_clock, pit_policy=policy))
    pit_rows.append({'policy':policy, 'clock':historical_clock,
        'visible_reports':len(selected.frame),
        'limitations':'; '.join(selected.context.get('limitations', ()))})
display(pd.DataFrame(pit_rows))
assert pit_rows[0]['visible_reports'] == 0
assert pit_rows[1]['visible_reports'] == 0
print('今天收到的终态历史不会回填到 2026-06-01 的 operational/safe 信息集。')


,policy,clock,visible_reports,limitations
0,operational_pit_v1,2026-06-01T09:30:00+08:00,0,report versions use declared supplier announce...
1,market_pit_safe_v1,2026-06-01T09:30:00+08:00,0,7 event revisions lacked revision-bound public...
2,best_effort_vendor_v1,2026-06-01T09:30:00+08:00,6,vendor event availability is a declared assump...


今天收到的终态历史不会回填到 2026-06-01 的 operational/safe 信息集。


In [14]:
# The source field is year-to-date cumulative revenue; derive only after PIT selection.
unit = income_events.field_meta['total_revenue']['unit']
basis = income_events.field_meta['total_revenue']['basis']
quarter = single_quarter(income_events, field='total_revenue', endpoint='income',
    report_type='1', basis=basis, unit=unit)
trailing = ttm(income_events, field='total_revenue', endpoint='income',
    report_type='1', basis=basis, unit=unit)
display(quarter.frame)
display(trailing.frame)
assert quarter.context['derivation']['input_snapshot_id'] == FULL_SNAPSHOT
assert trailing.field_meta['total_revenue']['basis'] == 'trailing_twelve_months'
statuses = pd.DataFrame([{'period':p['report_period'], 'status':p['status'],
    'missing_reason':p['missing_reason'], 'components':len(p['components'])}
    for p in trailing.field_meta['total_revenue']['by_key']])
display(statuses)
print('缺季度时结果保持空值；单季和 TTM 均保存组成报告与固定版本。')


,security_id,endpoint,report_type,report_period,total_revenue
0,cnstock.600000.SH.19991110,income,1,2024-12-31,<NA>
1,cnstock.600000.SH.19991110,income,1,2025-03-31,45922000000.0
2,cnstock.600000.SH.19991110,income,1,2025-06-30,44637000000.0
3,cnstock.600000.SH.19991110,income,1,2025-09-30,41721000000.0
4,cnstock.600000.SH.19991110,income,1,2025-12-31,41684000000.0
5,cnstock.600000.SH.19991110,income,1,2026-03-31,46573000000.0
6,cnstock.600000.SH.19991110,income,1,2026-06-30,47204000000.0


,security_id,endpoint,report_type,report_period,total_revenue
0,cnstock.600000.SH.19991110,income,1,2024-12-31,<NA>
1,cnstock.600000.SH.19991110,income,1,2025-03-31,<NA>
2,cnstock.600000.SH.19991110,income,1,2025-06-30,<NA>
3,cnstock.600000.SH.19991110,income,1,2025-09-30,<NA>
4,cnstock.600000.SH.19991110,income,1,2025-12-31,173964000000.0
5,cnstock.600000.SH.19991110,income,1,2026-03-31,174615000000.0
6,cnstock.600000.SH.19991110,income,1,2026-06-30,177182000000.0


,period,status,missing_reason,components
0,2024-12-31,source_missing,missing_quarter,7
1,2025-03-31,source_missing,missing_quarter,7
2,2025-06-30,source_missing,missing_quarter,7
3,2025-09-30,source_missing,missing_quarter,7
4,2025-12-31,value,None,7
5,2026-03-31,value,None,7
6,2026-06-30,value,None,7


缺季度时结果保持空值；单季和 TTM 均保存组成报告与固定版本。


In [15]:
# Each event stays at its native economic key; no daily forward fill.
examples = (
    ('balance_sheet_events', ('total_assets','total_liabilities'), {'endpoint':'balancesheet','report_type':'1'}),
    ('cash_flow_events', ('operating_cash_flow',), {'endpoint':'cashflow','report_type':'1'}),
    ('financial_indicator_events', ('roe','debt_to_assets'), {'endpoint':'fina_indicator'}),
    ('corporate_actions', ('process_status','cash_dividend_before_tax_per_share','ex_date'), {}),
    ('top_holders_reports', ('holders','holder_count','group_completeness','top10_ratio'), {}),
)
event_results = {}

In [16]:
for domain_name, requested, filters in examples:
    segment = full_manifest['domains'][domain_name]
    rows = [r for p in segment['partitions']
            for r in full.store.read_partition(p).to_pylist()]
    if not rows:
        print(domain_name, '当前范围无事件；保留真实空结果')
        continue
    selected_id = bank_id if any(r['security_id'] == bank_id for r in rows) else rows[0]['security_id']
    rows = [r for r in rows if r['security_id'] == selected_id]
    date_field = 'report_period'
    dates = sorted(str(r[date_field]) for r in rows)
    fields = tuple(f for f in requested if f in segment['contract']['fields'] and f not in segment['contract']['logical_key'])
    if not fields:
        print(domain_name, '无可请求的非键字段')
        continue
    q = EventQuery(domain_name, fields, (selected_id,), dates[0], dates[-1], cutoff_now,
        'operational_pit_v1', date_field, filters)
    result = full.events(snapshot=FULL_SNAPSHOT, query=q)
    event_results[domain_name] = result
    print(domain_name, 'security:', selected_id, 'rows:', len(result.frame),
          'key:', segment['contract']['logical_key'])
    display(result.frame.head(12))
    print('limitation:', result.context.get('limitations'))

balance_sheet_events security: cnstock.600000.SH.19991110 rows: 6 key: ['security_id', 'endpoint', 'report_type', 'report_period']


,security_id,endpoint,report_type,report_period,total_assets,total_liabilities
0,cnstock.600000.SH.19991110,balancesheet,1,2024-12-31,9461880000000.0,8717099000000.0
1,cnstock.600000.SH.19991110,balancesheet,1,2025-03-31,9552276000000.0,8796647000000.0
2,cnstock.600000.SH.19991110,balancesheet,1,2025-06-30,9645791000000.0,8844539000000.0
3,cnstock.600000.SH.19991110,balancesheet,1,2025-09-30,9892214000000.0,9051370000000.0
4,cnstock.600000.SH.19991110,balancesheet,1,2025-12-31,10081746000000.0,9257316000000.0
5,cnstock.600000.SH.19991110,balancesheet,1,2026-03-31,10305646000000.0,9464247000000.0


limitation: ['report versions use declared supplier announcement days; same-day corrections use actual observation order, not a verified publication sequence']


cash_flow_events security: cnstock.600000.SH.19991110 rows: 4 key: ['security_id', 'endpoint', 'report_type', 'report_period']


,security_id,endpoint,report_type,report_period,operating_cash_flow
0,cnstock.600000.SH.19991110,cashflow,1,2024-12-31,-333654000000.0
1,cnstock.600000.SH.19991110,cashflow,1,2025-06-30,21257000000.0
2,cnstock.600000.SH.19991110,cashflow,1,2025-12-31,375836000000.0
3,cnstock.600000.SH.19991110,cashflow,1,2026-03-31,91724000000.0


...

limitation: ['report versions use declared supplier announcement days; same-day corrections use actual observation order, not a verified publication sequence']
corporate_actions security: cnstock.600000.SH.19991110 rows: 9 key: ['security_id', 'report_period', 'announcement_date', 'process_status']


,security_id,report_period,announcement_date,process_status,cash_dividend_before_tax_per_share,ex_date
0,cnstock.600000.SH.19991110,2024-12-31,2025-03-29,股东大会通过,0.41,None
1,cnstock.600000.SH.19991110,2024-12-31,2025-03-29,预案,0.41,None
2,cnstock.600000.SH.19991110,2024-12-31,2025-06-28,实施,0.41,2025-07-16
3,cnstock.600000.SH.19991110,2025-06-30,2025-08-28,股东大会通过,<NA>,None
4,cnstock.600000.SH.19991110,2025-06-30,2025-08-28,预案,0.0,None
5,cnstock.600000.SH.19991110,2025-12-31,2026-03-31,预案,0.42,None
6,cnstock.600000.SH.19991110,2025-12-31,2026-06-27,实施,0.42,2026-07-16
7,cnstock.600000.SH.19991110,2026-06-30,2026-08-28,股东大会通过,<NA>,None
8,cnstock.600000.SH.19991110,2026-06-30,2026-08-28,预案,0.0,None


limitation: ['terminal states ordered by system observation under declared source policy; vendor revision/publication order is unknown']
top_holders_reports security: cnstock.600000.SH.19991110 rows: 6 key: ['security_id', 'report_period']


,security_id,report_period,group_completeness,holders,holder_count,top10_ratio
0,cnstock.600000.SH.19991110,2025-03-31,supplier_report_complete,"[{""hold_amount_shares"":1395571025.0,""hold_rati...",10,76.2265
1,cnstock.600000.SH.19991110,2025-06-30,supplier_report_complete,"[{""hold_amount_shares"":1395571025.0,""hold_rati...",10,75.3584
2,cnstock.600000.SH.19991110,2025-09-30,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,74.6127
3,cnstock.600000.SH.19991110,2025-12-31,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,75.2567
4,cnstock.600000.SH.19991110,2026-03-31,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,74.9193
5,cnstock.600000.SH.19991110,2026-06-30,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,74.8702


limitation: ['report versions use declared supplier announcement days; same-day corrections use actual observation order, not a verified publication sequence']


In [17]:
actions = event_results.get('corporate_actions')
if actions is not None:
    statuses = actions.frame['process_status'].value_counts(dropna=False).rename_axis('process_status').reset_index(name='rows')
    display(statuses)
    implemented = actions.frame.loc[actions.frame['process_status'].eq('实施')]
    print('明确实施状态行:', len(implemented), '；其他状态不可直接当作已实施现金流。')
holders = event_results.get('top_holders_reports')
if holders is not None:
    display(holders.frame[['report_period','holder_count','group_completeness','top10_ratio']])
    assert all(pd.isna(value) for value in holders.frame.loc[
        ~holders.frame['group_completeness'].eq('supplier_report_complete'), 'top10_ratio'])
    print('supplier_report_complete 只表示当前来源响应满足十名不同持有人及字段检查；不是独立全量认证。')
print('价格涨跌停为当日未复权边界，不代表一定可成交。')
print('预测类数据不在这次 FullSourcePlan 中，不能把缺域解释为零预测。')


,process_status,rows
0,预案,4
1,股东大会通过,3
2,实施,2


明确实施状态行: 2 ；其他状态不可直接当作已实施现金流。


,report_period,holder_count,group_completeness,top10_ratio
0,2025-03-31,10,supplier_report_complete,76.2265
1,2025-06-30,10,supplier_report_complete,75.3584
2,2025-09-30,10,supplier_report_complete,74.6127
3,2025-12-31,10,supplier_report_complete,75.2567
4,2026-03-31,10,supplier_report_complete,74.9193
5,2026-06-30,10,supplier_report_complete,74.8702


supplier_report_complete 只表示当前来源响应满足十名不同持有人及字段检查；不是独立全量认证。
价格涨跌停为当日未复权边界，不代表一定可成交。
预测类数据不在这次 FullSourcePlan 中，不能把缺域解释为零预测。


In [18]:
# A real native implementation row has a later implementation announcement.
action_id = full_scope['identity_map']['000001.SZ']
def implemented_dividend_at(cutoff, policy):
    return full.events(snapshot=FULL_SNAPSHOT,
        query=EventQuery('corporate_actions',
            ('cash_dividend_before_tax_per_share',
             'implementation_announcement_date','record_date','ex_date'),
            (action_id,), '2026-05-23', '2026-05-23', cutoff, policy,
            'announcement_date', {'process_status':'实施'}))

early = implemented_dividend_at('2026-06-01T09:30:00+08:00',
    'best_effort_vendor_v1')
after_implementation = implemented_dividend_at('2026-06-08T09:30:00+08:00',
    'best_effort_vendor_v1')

In [19]:
strict_after = implemented_dividend_at('2026-06-08T09:30:00+08:00',
    'operational_pit_v1')
display(pd.DataFrame([
    {'cutoff':'2026-06-01 09:30','policy':'best_effort_vendor_v1',
     'visible_implementation_rows':len(early.frame)},
    {'cutoff':'2026-06-08 09:30','policy':'best_effort_vendor_v1',
     'visible_implementation_rows':len(after_implementation.frame)},
    {'cutoff':'2026-06-08 09:30','policy':'operational_pit_v1',
     'visible_implementation_rows':len(strict_after.frame)},
]))
assert len(early.frame) == 0
assert len(after_implementation.frame) == 1
assert len(strict_after.frame) == 0
implementation = after_implementation.frame.iloc[0]
assert str(implementation['implementation_announcement_date']) == '2026-06-05'

,cutoff,policy,visible_implementation_rows
0,2026-06-01 09:30,best_effort_vendor_v1,0
1,2026-06-08 09:30,best_effort_vendor_v1,1
2,2026-06-08 09:30,operational_pit_v1,0


In [20]:
assert str(implementation['record_date']) == '2026-06-11'
assert str(implementation['ex_date']) == '2026-06-12'
assert implementation['cash_dividend_before_tax_per_share'] == 0.36
display(after_implementation.frame)
print('原公告 05-23、实施公告 06-05；完整实施行不能在 06-01 预知。')
print('06-08 仅按供应商公告时间假设可见；严格 historical cutoff 仍早于原 Raw 接收。')

,security_id,report_period,announcement_date,process_status,cash_dividend_before_tax_per_share,implementation_announcement_date,record_date,ex_date
0,cnstock.000001.SZ.19910403,2025-12-31,2026-05-23,实施,0.36,2026-06-05,2026-06-11,2026-06-12


原公告 05-23、实施公告 06-05；完整实施行不能在 06-01 预知。
06-08 仅按供应商公告时间假设可见；严格 historical cutoff 仍早于原 Raw 接收。


## 7. 采集 plan、批量后处理和 checkpoint

执行入口是 `prepare → plan → run → verify → audit`。参考准备保留完整供应商成员与上市来源，debug 仅缩小后续事实证券范围。行情、因子和停复牌按交易日全市场请求，三表按公告月 VIP 批量请求，指标按报告季度请求；必须单股请求的股东来源仍按供应商接口执行。默认 8 个在途、300 次/分钟全局上限，主表另有限速。Raw 在响应收到时落盘，Canonical 按月批量写列。

plan 固定选择器、范围、重试、限速和 base Snapshot；checkpoint 保存已成功 Raw 与发布块。接口失败重试有界，明确截断的响应要拆分；外部事实差异只提示 warning。中断后用同一 plan/operation 继续，成功接收时间保留，整作业完成后推进 current。

In [21]:
import subprocess
cli = [sys.executable, '-m', 'axiom_data', '--data-root', str(ROOT)]
status = json.loads(subprocess.check_output(cli + ['status','--plan',str(ROOT.with_suffix('.plan.json'))], text=True))
display(pd.DataFrame([{k:status[k] for k in ('status','completed_requests','planned_requests','trading_sessions','total_chunks')}]))
assert status['status'] == 'success' and status['completed_requests'] == 735
print('读取已完成作业状态；没有启动采集。')


,status,completed_requests,planned_requests,trading_sessions,total_chunks
0,success,735,735,242,13


读取已完成作业状态；没有启动采集。


In [22]:
from axiom_data.full_sources import FullSourcePlan, full_source_status, verify_full_sources
full_plan_path = Path(os.environ.get('AXIOM_UNIFIED_PLAN',
    REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.plan.json'))
plan_envelope = json.loads(full_plan_path.read_text())
full_plan = FullSourcePlan.from_dict(plan_envelope['job'])
full_status = full_source_status(full.store, plan=full_plan,
    operation_id=plan_envelope['operation_id'])
display(pd.DataFrame([{k:full_status.get(k) for k in ('status','phase','planned_event_requests',
    'completed_event_requests','event_raw_rows','event_raw_attempts','elapsed_seconds')}]))
assert full_status['status'] == 'success'
verified_full = verify_full_sources(full.store, plan=full_plan,
    operation_id=plan_envelope['operation_id'])
assert verified_full['verified'] and verified_full['snapshot_id'] == FULL_SNAPSHOT
print('独立验证:', verified_full)

,status,phase,planned_event_requests,completed_event_requests,event_raw_rows,event_raw_attempts,elapsed_seconds
0,success,complete,741,741,591007,741,152.311628


独立验证: {'verified': True, 'snapshot_id': 's_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049', 'market_verified': True, 'calendar_verified': True, 'selected_event_raw_batches': 741, 'unchanged_event_raw_batches': 331, 'event_raw_rows': 591007, 'coverage_basis': 'selected Raw requests, published facts and verified unchanged receipts; supplier completeness unverified'}


In [23]:
full_audit_path = Path(os.environ.get('AXIOM_UNIFIED_AUDIT',
    REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.audit.json'))
full_audit = json.loads(full_audit_path.read_text())
assert full_audit['snapshot_id'] == FULL_SNAPSHOT and full_audit['status'] == 'passed'
assert not full_audit['issues']
print('真实完整源 audit:', full_audit['status'], 'issues:', len(full_audit['issues']),
      'elapsed_seconds:', full_audit['elapsed_seconds'])
full_verify_path = Path(os.environ.get('AXIOM_UNIFIED_VERIFY',
    REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.verify.json'))
full_verify = json.loads(full_verify_path.read_text())
assert full_verify['snapshot_id'] == FULL_SNAPSHOT and full_verify['verified']
print('固定计划离线 verify:', full_verify['verified'],
      '；供应商覆盖限制:', full_verify['coverage_basis'])
nochange_path = REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.nochange.json'

真实完整源 audit: passed issues: 0 elapsed_seconds: 5.193
固定计划离线 verify: True ；供应商覆盖限制: selected Raw requests, published facts and verified unchanged receipts; supplier completeness unverified


In [24]:
if nochange_path.exists():
    nochange = json.loads(nochange_path.read_text())
    assert nochange['snapshot_id'] == FULL_SNAPSHOT and nochange['status'] == 'passed'
    print('完成作业同计划重放:', nochange)

完成作业同计划重放: {'basis': 'completed operation checkpoint returned without client source calls', 'raw_records_after': 969, 'raw_records_before': 969, 'snapshot_id': 's_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049', 'status': 'passed', 'supplier_requests_on_completed_replay': 0}


## 8. Daily update 与历史 refresh

新增交易日使用 `mode=daily` 和原稳定身份绑定，固定上一版作 base；参考准备携带上一次 preparation，收新 dated 成员快照才更新区间。财务默认回看 31 个原公告日。供应商更正若涉及更早公告日或报告期，另建明确历史 refresh 计划，只补相关源范围；新响应有新的真实接收时间。

bulk 小样本验证采集链路，daily 的空重复/晚到与发布语义由独立测试和既有真实增量证据验证；此次没有宣称在生产十二年根实际跑过 daily。股票历史并集和 ETF 的十二年配置均可直接交给 CLI；它们不依赖 agent，也不调用模型 API。

```sh

axiom-data --data-root /absolute/data-12y prepare --config examples/delivery_csi1800_twelve_year.json --output-scope delivery/csi1800-twelve-year.scope.json --token-file /absolute/private/tushare-token.txt
axiom-data --data-root /absolute/data-12y plan --scope delivery/csi1800-twelve-year.scope.json --operation-id csi1800-201411-202609-bulk --output delivery/csi1800-twelve-year.plan.json
axiom-data --data-root /absolute/data-12y run --plan delivery/csi1800-twelve-year.plan.json --token-file /absolute/private/tushare-token.txt
axiom-data --data-root /absolute/data-12y status --plan delivery/csi1800-twelve-year.plan.json
axiom-data --data-root /absolute/data-12y verify --plan delivery/csi1800-twelve-year.plan.json
axiom-data --data-root /absolute/data-12y audit --plan delivery/csi1800-twelve-year.plan.json --snapshot current --output delivery/csi1800-twelve-year.audit.json
```

参数与源修订窗口详见 [CLI](../docs/cli.md)。

## 9. 生命周期与月度成员边界

上市、退市日期和状态采用 Tushare；不接官网补写生产数据。成员延续的是最近可见完整供应商快照，session、snapshot_date 和 Raw 收到时间分开。没有原历史观察的 strict 查询返回未知，best-effort 才采用明确的历史供应商假设。下面用真实主表与成员显示这些边界。

In [25]:
# Production listing and status facts come from the pinned Tushare Snapshot.
listing_manifest = UNIFIED.store.load_snapshot(UNIFIED_SNAPSHOT)
master_domain = listing_manifest['domains']['security_master']
assert master_domain['source_profile']['id'].startswith('tushare.')
master = pd.DataFrame([r for part in master_domain['partitions']
    for r in UNIFIED.store.read_partition(part).to_pylist()])
listing_columns = [c for c in ('security_id','listing_date','delisting_date',
    'vendor_delist_date','list_status','first_observed_at') if c in master.columns]
display(master[listing_columns].head(8))
print('生产上市与退市来源:', master_domain['source_profile']['id'])

,security_id,listing_date,delisting_date,vendor_delist_date,list_status,first_observed_at
0,cnstock.600000.SH.19991110,1999-11-10,None,None,L,2026-09-28 13:16:33.564276+00:00
1,cnstock.000001.SZ.19910403,1991-04-03,None,None,L,2026-09-28 13:16:37.320913+00:00


生产上市与退市来源: tushare.local.stock_basic.v1


In [26]:
if 'listing_events' in listing_manifest['domains']:
    event_domain = listing_manifest['domains']['listing_events']
    assert event_domain['source_profile']['id'].startswith('tushare.')
    events = pd.DataFrame([r for part in event_domain['partitions']
        for r in UNIFIED.store.read_partition(part).to_pylist()])
    if not events.empty:
        event_columns = [c for c in ('security_id','listing_date','delisting_date',
            'event_type','first_observed_at') if c in events.columns]
        delisted_events = (events.loc[events.delisting_date.notna()]
            if 'delisting_date' in events else events)
        display(delisted_events[event_columns].head(8))
        print('规范上市事件来源:', event_domain['source_profile']['id'])
state_day = '2026-08-31'
state_ids = (IDS['000001.SZ'], IDS['600000.SH'])

,security_id,listing_date,delisting_date,event_type,first_observed_at
0,cnstock.000508.SZ.19930430,1993-04-30,1999-07-12,delisting,2026-09-28 13:16:38.388413+00:00
1,cnstock.600625.SH.19930106,1993-01-06,2001-04-23,delisting,2026-09-28 13:16:34.649002+00:00
2,cnstock.000588.SZ.19960123,1996-01-23,2001-06-15,delisting,2026-09-28 13:16:38.388413+00:00
3,cnstock.000015.SZ.19920625,1992-06-25,2001-10-22,delisting,2026-09-28 13:16:38.388413+00:00
4,cnstock.000556.SZ.19940525,1994-05-25,2002-05-29,delisting,2026-09-28 13:16:38.388413+00:00
5,cnstock.000003.SZ.19910703,1991-07-03,2002-06-14,delisting,2026-09-28 13:16:38.388413+00:00
6,cnstock.000675.SZ.19961226,1996-12-26,2002-08-20,delisting,2026-09-28 13:16:38.388413+00:00
7,cnstock.000689.SZ.19961231,1996-12-31,2002-09-05,delisting,2026-09-28 13:16:38.388413+00:00


规范上市事件来源: tushare.stock_basic.listing_events.v1


In [27]:
state_q = QuerySpec('market_daily', ('close',), state_ids, (state_day,),
    'best_effort_vendor_v1', {state_day:state_day+'T20:30:00+08:00'})
production_states = UNIFIED.states(snapshot=UNIFIED_SNAPSHOT, query=state_q)
display(production_states.frame)
past_states = UNIFIED.states(snapshot=UNIFIED_SNAPSHOT,
    query=replace(state_q, pit_policy='operational_pit_v1'))
display(past_states.frame)
assert past_states.frame.market_state.eq('unknown_status').all()
print('交易状态依据留存供应商事实和日历推导；外部交叉校验只能提示冲突，不改写生产值。')
print('严格历史 cutoff 早于原 Raw 接收，状态仍未知。')

,security_id,session,market_state,close
0,cnstock.000001.SZ.19910403,2026-08-31,unknown_status,11.72
1,cnstock.600000.SH.19991110,2026-08-31,unknown_status,9.16


,security_id,session,market_state,close
0,cnstock.000001.SZ.19910403,2026-08-31,unknown_status,<NA>
1,cnstock.600000.SH.19991110,2026-08-31,unknown_status,<NA>


交易状态依据留存供应商事实和日历推导；外部交叉校验只能提示冲突，不改写生产值。
严格历史 cutoff 早于原 Raw 接收，状态仍未知。


In [28]:
# A Tushare month-end snapshot is observed on its own trade_date.
MEMBER_SNAPSHOT = UNIFIED_SNAPSHOT
member_manifest = UNIFIED.store.load_snapshot(MEMBER_SNAPSHOT)
member_domain = member_manifest['domains']['universe_membership']
member_coverage = member_domain['coverage']
assert member_coverage['basis'] == 'tushare_dated_index_weight_snapshot_v1'
assert member_coverage['between_snapshots'] == 'carry_forward'
print('生产成员来源:', member_domain['source_profile']['id'])
print('固定 Snapshot:', MEMBER_SNAPSHOT,
      '；规范成员行:', sum(p['rows'] for p in member_domain['partitions']))
print('成员覆盖摘要:', {k:v for k,v in member_coverage.items()
    if k in ('basis','between_snapshots','universe_ids','observation_dates','limitations')})
state_rows = pd.DataFrame(member_coverage.get('complete_states', ()))

生产成员来源: tushare.index_weight.dated_membership.v1
固定 Snapshot: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049 ；规范成员行: 3874
成员覆盖摘要: {'basis': 'tushare_dated_index_weight_snapshot_v1', 'between_snapshots': 'carry_forward', 'limitations': ['Supplier index_weight was observed on dated snapshots only; no intramonth membership change is proven.', 'Best-effort uses an assumed 18:00 Asia/Shanghai dated release; strict policies use actual Raw receipt.']}


In [29]:
if not state_rows.empty:
    selected = (state_rows.loc[state_rows.universe_id.eq('csi1800')]
        if 'universe_id' in state_rows else state_rows)
    display(selected)

MEMBER_DAYS = ('2026-07-31','2026-08-03','2026-08-31')
vendor_query = QuerySpec('universe_membership', ('is_member',), (SYMBOL,),
    MEMBER_DAYS, 'best_effort_vendor_v1',
    {day:day+'T20:30:00+08:00' for day in MEMBER_DAYS}, universe_id='csi1800')
vendor_members = UNIFIED.members(snapshot=MEMBER_SNAPSHOT, query=vendor_query)
display(vendor_members.frame)
member_meta = pd.DataFrame(vendor_members.field_meta['is_member']['by_key'])
meta_columns = [c for c in ('security_id','session','snapshot_date','source_snapshot_date',
    'effective_from','first_observed_at','availability_basis','missing_reason')
    if c in member_meta.columns]

,complete,dependency_raw_batch_ids,effective_from,effective_to,first_observed_at,member_count,member_set_sha256,member_set_source,raw_batch_id,revision_id,revision_sequence,source_snapshot_date,state_id,universe_id
12,True,"[b_53fad64807cc41448f028719847d2ee9, b_7277acc...",2026-05-29,2026-06-30,2026-09-28T13:17:24.908931+00:00,1800,4b9d1d17ccd9875474ec3f10369699179b5cdf2ef98b06...,canonical_intervals_v1,b_53fad64807cc41448f028719847d2ee9,vendor:9f0d23f6e5910baacf346b1379116ef805178b2...,1,2026-05-29,vendor:csi1800:2026-05-29,csi1800
13,True,"[b_b79a4a5d683548aaa11803d5edf648f3, b_1c3c1f6...",2026-06-30,2026-07-31,2026-09-28T13:17:26.380734+00:00,1800,869ca57470d399de6c11b1e3db3cde9d8b14fc705d1072...,canonical_intervals_v1,b_b79a4a5d683548aaa11803d5edf648f3,vendor:a8910eb6904cefcb0971b3a80d0349bd5cde133...,1,2026-06-30,vendor:csi1800:2026-06-30,csi1800
14,True,"[b_18fbd2b97d7440b8b8b17f96bdee23e5, b_4fe02a3...",2026-07-31,2026-08-31,2026-09-28T13:17:27.310742+00:00,1800,869ca57470d399de6c11b1e3db3cde9d8b14fc705d1072...,canonical_intervals_v1,b_18fbd2b97d7440b8b8b17f96bdee23e5,vendor:091f1143725c1fd73de240279684fd5e213a87c...,1,2026-07-31,vendor:csi1800:2026-07-31,csi1800
15,True,"[b_ccedf03a766d41c89d7e292efc276403, b_d2c26e2...",2026-08-31,None,2026-09-28T13:17:28.534869+00:00,1800,869ca57470d399de6c11b1e3db3cde9d8b14fc705d1072...,canonical_intervals_v1,b_ccedf03a766d41c89d7e292efc276403,vendor:1fe9cb9e4b4acc1c7443ff96c256dc5d26ed1c9...,1,2026-08-31,vendor:csi1800:2026-08-31,csi1800


,security_id,session,is_member
0,cnstock.000001.SZ.19910403,2026-07-31,True
1,cnstock.000001.SZ.19910403,2026-08-03,True
2,cnstock.000001.SZ.19910403,2026-08-31,True


In [30]:
display(member_meta[meta_columns])
snapshot_by_session = member_meta.set_index('session')['source_snapshot_date']
assert snapshot_by_session.loc['2026-08-03'] == '2026-07-31'
assert snapshot_by_session.loc['2026-08-31'] == '2026-08-31'
observed = vendor_members.frame.set_index('session').is_member
assert pd.notna(observed.loc['2026-07-31'])
assert observed.loc['2026-08-03'] == observed.loc['2026-07-31']
strict = UNIFIED.members(snapshot=MEMBER_SNAPSHOT,
    query=replace(vendor_query, pit_policy='operational_pit_v1'))
display(strict.frame)
assert strict.frame.is_member.isna().all()
print('08-03 沿用最近的 07-31 供应商快照；08-31 新快照不倒填 8 月初。')
print('这只是 Tushare 供应商研究口径；月内真实调样时刻没有逐日来源。')
CHANGED_ID = 'cnstock.000506.SZ.19930312'
CHANGE_DAYS = ('2026-05-29','2026-06-01','2026-06-30','2026-07-01')

,security_id,session,source_snapshot_date,first_observed_at,availability_basis,missing_reason
0,cnstock.000001.SZ.19910403,2026-07-31,2026-07-31,2026-09-28T13:17:24.908931+00:00,declared_vendor_assumption,None
1,cnstock.000001.SZ.19910403,2026-08-03,2026-07-31,2026-09-28T13:17:24.908931+00:00,declared_vendor_assumption,None
2,cnstock.000001.SZ.19910403,2026-08-31,2026-08-31,2026-09-28T13:17:24.908931+00:00,declared_vendor_assumption,None


,security_id,session,is_member
0,cnstock.000001.SZ.19910403,2026-07-31,<NA>
1,cnstock.000001.SZ.19910403,2026-08-03,<NA>
2,cnstock.000001.SZ.19910403,2026-08-31,<NA>


08-03 沿用最近的 07-31 供应商快照；08-31 新快照不倒填 8 月初。
这只是 Tushare 供应商研究口径；月内真实调样时刻没有逐日来源。


In [31]:
change_query = QuerySpec('universe_membership', ('is_member',), (CHANGED_ID,),
    CHANGE_DAYS, 'best_effort_vendor_v1',
    {day:day+'T20:30:00+08:00' for day in CHANGE_DAYS}, universe_id='csi1800')
changed = UNIFIED.members(snapshot=MEMBER_SNAPSHOT, query=change_query)
display(changed.frame)
assert changed.frame.is_member.tolist() == [False,False,True,True]
change_meta = pd.DataFrame(changed.field_meta['is_member']['by_key'])
display(change_meta[['session','source_snapshot_date','first_observed_at']])
assert change_meta.source_snapshot_date.tolist() == [
    '2026-05-29','2026-05-29','2026-06-30','2026-06-30']
print('000506 仅在供应商 06-30 新快照转为成员；没有倒填为 06-15 的精确调整。')

,security_id,session,is_member
0,cnstock.000506.SZ.19930312,2026-05-29,False
1,cnstock.000506.SZ.19930312,2026-06-01,False
2,cnstock.000506.SZ.19930312,2026-06-30,True
3,cnstock.000506.SZ.19930312,2026-07-01,True


,session,source_snapshot_date,first_observed_at
0,2026-05-29,2026-05-29,2026-09-28T13:17:24.908931+00:00
1,2026-06-01,2026-05-29,2026-09-28T13:17:24.908931+00:00
2,2026-06-30,2026-06-30,2026-09-28T13:17:26.380734+00:00
3,2026-07-01,2026-06-30,2026-09-28T13:17:26.380734+00:00


000506 仅在供应商 06-30 新快照转为成员；没有倒填为 06-15 的精确调整。


In [32]:
# This is a saved live Tushare probe, independent of the tutorial's production Snapshot.
cadence = json.loads((REPO / 'docs/tushare-membership-cadence.json').read_text())
probe_rows = []
for item in cadence['results']:
    params = item['params']
    probe_rows.append({'index_code':params['index_code'],
        'requested_from':params['start_date'], 'requested_to':params['end_date'],
        'rows':item['rows'], 'returned_trade_dates':','.join(item['trade_date_counts']),
        'raw_batch_id':item['raw_batch_id'], 'observed_at':item['observed_at']})
probe = pd.DataFrame(probe_rows)
display(probe)
assert len(probe) == 12
for index_code, expected in (('000300.SH',300),('000905.SH',500),('000852.SH',1000)):
    segment = probe.loc[probe.index_code.eq(index_code)]
    assert segment.loc[segment.requested_from.isin(('20260612','20260615')), 'rows'].eq(0).all()
    assert segment.loc[segment.requested_from.eq('20260630'), 'rows'].iloc[0] == expected
    assert segment.loc[segment.requested_from.eq('20260601'), 'returned_trade_dates'].iloc[0] == '20260630'
print('实际来源仅在六月末返回同日名单；逐日请求不能制造逐日事实。')
MONTH_ENDS = ('2026-07-31', '2026-08-31')


,index_code,requested_from,requested_to,rows,returned_trade_dates,raw_batch_id,observed_at
0,000300.SH,20260612,20260612,0,,b_b382f8701049473c9645468637c02f24,2026-10-03T05:02:05.724878+00:00
1,000300.SH,20260615,20260615,0,,b_91cc3b1bfd944b989ce71f08c663650a,2026-10-03T05:02:06.109068+00:00
2,000300.SH,20260630,20260630,300,20260630,b_862fbc2c4a41440c8598c7d9aeeae859,2026-10-03T05:02:06.507870+00:00
3,000300.SH,20260601,20260630,300,20260630,b_14e33c6e9be941bcb0408e2def4813ab,2026-10-03T05:02:06.884449+00:00
4,000905.SH,20260612,20260612,0,,b_bce58844823b4a78b950182079d2d911,2026-10-03T05:02:07.277555+00:00
5,000905.SH,20260615,20260615,0,,b_bf0f69de259d4d5f91afb1fad56d0063,2026-10-03T05:02:07.652712+00:00
6,000905.SH,20260630,20260630,500,20260630,b_9613b19b682d47aeaadd3f04ed526ad3,2026-10-03T05:02:08.045632+00:00
7,000905.SH,20260601,20260630,500,20260630,b_3bb98aaff5884434810c8de3fcec7bad,2026-10-03T05:02:08.437697+00:00
8,000852.SH,20260612,20260612,0,,b_d2386f37ddbb4d78a5a5248c136072b8,2026-10-03T05:02:08.886354+00:00
9,000852.SH,20260615,20260615,0,,b_bd34b0aafbd248a398cf57325ff24ae0,2026-10-03T05:02:09.245978+00:00


实际来源仅在六月末返回同日名单；逐日请求不能制造逐日事实。


## 10. 数据和代码变化时，哪些工作需要重做

先界定改动涉及哪个域、字段、日期和读取规则，再选择最小动作。财务按报告期和披露时间判断，不能仅按行情 session 判断影响范围。

| 实际问题 | 当前最小流程 | 已有研究如何处理 |
|---|---|---|
| 用已有 close 计算新的动量 | Reader 读取已存字段，Research 增加配方与参数；无需补拉或重建事实 | 原研究照常保留；新配方有自己的代码版本 |
| 新增供应商字段或接口 | 声明键、类型、单位、时间和来源映射。Raw 已含字段时离线重建；没采到时只补这个接口和窗口 | 用到新字段的研究保存新 Snapshot；原查询仍可读取原版本 |
| 原 Raw 漏拉、请求错误或供应商后来修订 | 补发对应选择器，追加真实接收记录；缺文件先从已验证副本恢复 | 记录错误范围和受影响数据引用，相关研究标为待复核 |
| 归一化、复权、PIT 或 schema 有错误/变化 | 归一化用修正版重建所选域；Reader/派生错误用修正版重读、重算；不兼容格式另做明确迁移 | 固定旧结果供对照，重跑实际受影响的实验；保留旧代码才能重现旧行为 |

`rebuild` 当前替换**整个所选域**，输入必须是这个域所需的完整 Raw 集合。标准 CLI 自动收集指定 Snapshot 引用的 Raw；高级 `--request` 只给一天 Raw 会把该域重建成一天，并没有自动分区修补能力。未选域和旧 Snapshot 保持不变。增加任意新 endpoint 仍需写小 adapter、合同、映射和对应验证，不会由同逻辑 rebuild 自动完成。

有日志半写时，Reader 仅读最后完整行之前的记录。恢复写入时先保留未完成尾片，再截到完整行边界；已完成记录的接收时间不变。已完成行损坏或对象摘要错误仍报错，需恢复正确字节或补采。新采样只能证明新观察，无法还原丢失的历史 vintage。

**可以复现的结果也可能算错。** 发现错误后，在修正记录中写清域/字段/窗口、错误版本、原因、替代版本和复核状态，关联到使用这些输入的实验。保留旧输出便于审计，同时标记它需要重算；其他未受影响实验继续有效。目前没有自动分析所有实验依赖的全局失效系统，也不需要先建设一个。完整操作边界见[变更与修复](../docs/data-change-and-recovery.md)。

后来重复 fetch 的 Raw 可能保存新增字段而未产生新 Snapshot。历史字段扩展先用 `data.select_raw(domains=("market_daily",), receipt_cutoff="2026-10-03T12:00:00Z")` 预览并保存成功/空响应 ID，再显式整域 rebuild；原请求没收到字段仍为 null。默认 bundle 仅保存 Snapshot 及祖先闭包；完整 Raw 备份另用 `export --raw-backup-cutoff`，不包括作业 checkpoints。

实际 builder 在 `build_context.builder`，来自干净 commit 或留存并核对的安装 wheel；同时绑定依赖锁、环境和操作配置。dirty 源码拒绝构建。新建域更新 builder，未变域保持原来源；供应商输入配置与实际代码来源分别保存。


## 11. 故障恢复与搬移

Raw 半写的完整前缀读写已通过专项测试：只隔离末尾未完成片段，不改成功接收记录。完整行损坏仍需副本恢复。下面真实全年行情从同 Raw 重建所选域，其他域保持，current 不动；统一两证券样本此前已完成全域搬移与重建，逐域比较文件字节、合同和成员读取。

这些是同一机器换目录与隔离环境的实际验证；跨操作系统是另外的部署验收。bundle 保存源文件闭包、接收日志和选定代码依赖，不会把重新收到的历史值伪装为旧 vintage。

恢复先查原 operation 的阶段、失败块与已完成 Raw，再用同 plan 继续。已成功接收不重新请求；归一化失败的成功 Raw 用 `apply_saved_raw` 继续。没有源行/缺季度保留合法缺失；身份、单位和文件错误修正后才重跑对应操作。此处的反例使用临时根，实际正式根没有重新采集。

In [33]:
BUNDLE = Path(str(ROOT) + '-bundle')
LOCAL = Data(TEMP / 'relocated-market')
assert import_bundle(BUNDLE, LOCAL.store.root) == SNAPSHOT
base = LOCAL.resolve('current')
old = LOCAL.store.load_snapshot(base)
start = time.perf_counter()
rebuilt = LOCAL.rebuild(base_snapshot=base,
    raw_batch_ids=old['domains']['market_daily']['raw_batch_ids'], domains=['market_daily'],
    operation_id='tutorial-real-market-rebuild', build_context={'purpose':'real offline tutorial'}, promote=False)
new = LOCAL.store.load_snapshot(rebuilt.snapshot_id)
assert new['domains']['market_daily']['partitions'] == old['domains']['market_daily']['partitions']
assert all(new['domains'][k] == old['domains'][k] for k in old['domains'] if k != 'market_daily')
assert LOCAL.resolve('current') == base
print({'elapsed_seconds':round(time.perf_counter()-start,3),'market_partitions_byte_equal':True,
       'unrelated_domains_equal':True,'current_unchanged':True,'new_snapshot':rebuilt.snapshot_id})


{'elapsed_seconds': 23.525, 'market_partitions_byte_equal': True, 'unrelated_domains_equal': True, 'current_unchanged': True, 'new_snapshot': 's_7c138f72bb15630d19c4a4444854db6616d56c6b4c8a48cbc0841f57c11c0b12'}


In [34]:
# Same-host relocation and all-domain rebuild were completed from retained Tushare Raw.
portable_path = Path(os.environ.get('AXIOM_UNIFIED_PORTABLE_REPORT',
    REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.portable.json'))
portable = json.loads(portable_path.read_text())
assert portable['status'] == 'passed' and portable['source_snapshot'] == FULL_SNAPSHOT
assert portable['current_unchanged']
assert set(portable['domains']) == set(full_manifest['domains'])
# The formal portable report already checked the whole bundle and every partition.
# This cell checks the recorded result and independently rereads the moved data.
assert Path(portable['bundle']).exists()
restored_root = Path(os.environ.get('AXIOM_UNIFIED_RESTORED_ROOT', portable['restored_root']))
restored = Data(restored_root)
assert restored.resolve('current') == FULL_SNAPSHOT
imported_manifest = restored.store.load_snapshot(FULL_SNAPSHOT)
rebuilt_manifest = restored.store.load_snapshot(portable['rebuilt_snapshot'])

In [35]:
portable_rows = []
for domain_name, comparison in portable['domains'].items():
    same_contract = (full_manifest['domains'][domain_name]['contract'] ==
        imported_manifest['domains'][domain_name]['contract'] ==
        rebuilt_manifest['domains'][domain_name]['contract'])
    assert comparison['partitions_identical'] and comparison['profile_identical'] and same_contract
    portable_rows.append({'domain':domain_name,
        'rows':sum(p['rows'] for p in full_manifest['domains'][domain_name]['partitions']),
        'partition_bytes_identical':True, 'source_profile_identical':True,
        'contract_identical':same_contract,
        'coverage_identical':comparison['coverage_identical']})
display(pd.DataFrame(portable_rows))
assert portable['domains']['listing_events']['coverage_identical']
assert portable['domains']['universe_membership']['coverage_identical']
source_probe = vendor_members.frame[['security_id','session','is_member']].reset_index(drop=True)

,domain,rows,partition_bytes_identical,source_profile_identical,contract_identical,coverage_identical
0,adjustment_factors,130,True,True,True,False
1,balance_sheet_events,13,True,True,True,False
2,cash_flow_events,10,True,True,True,False
3,corporate_actions,20,True,True,True,False
4,financial_events,14,True,True,True,False
5,financial_indicator_events,12,True,True,True,False
6,listing_events,5888,True,True,True,True
7,market_daily,130,True,True,True,False
8,price_limits,130,True,True,True,False
9,security_master,2,True,True,True,False


In [36]:
imported_probe = restored.members(snapshot=FULL_SNAPSHOT,
    query=vendor_query).frame[['security_id','session','is_member']].reset_index(drop=True)
rebuilt_probe = restored.members(snapshot=portable['rebuilt_snapshot'],
    query=vendor_query).frame[['security_id','session','is_member']].reset_index(drop=True)
assert source_probe.equals(imported_probe) and source_probe.equals(rebuilt_probe)
display(source_probe)
source_change = changed.frame[['security_id','session','is_member']].reset_index(drop=True)
imported_change = restored.members(snapshot=FULL_SNAPSHOT,
    query=change_query).frame[['security_id','session','is_member']].reset_index(drop=True)
rebuilt_change = restored.members(snapshot=portable['rebuilt_snapshot'],
    query=change_query).frame[['security_id','session','is_member']].reset_index(drop=True)
assert source_change.equals(imported_change) and source_change.equals(rebuilt_change)
display(source_change)
print('同机换目录导入与 Raw 全域重建:', len(portable_rows), '个域；供应商成员原根/搬移根/重建根一致。')
print('覆盖记录的 chunk/request 分组可不同；上市和成员 coverage 相同。', portable['coverage_note'])

,security_id,session,is_member
0,cnstock.000001.SZ.19910403,2026-07-31,True
1,cnstock.000001.SZ.19910403,2026-08-03,True
2,cnstock.000001.SZ.19910403,2026-08-31,True


,security_id,session,is_member
0,cnstock.000506.SZ.19930312,2026-05-29,False
1,cnstock.000506.SZ.19930312,2026-06-01,False
2,cnstock.000506.SZ.19930312,2026-06-30,True
3,cnstock.000506.SZ.19930312,2026-07-01,True


同机换目录导入与 Raw 全域重建: 14 个域；供应商成员原根/搬移根/重建根一致。
覆盖记录的 chunk/request 分组可不同；上市和成员 coverage 相同。 Market and event execution coverage records different chunk versus request grouping in the offline rebuild; all 14 canonical partitions and source profiles are identical, and both vendor listing and membership coverage are identical.


## 12. Qlib 文件格式和实际消费者

exporter 只读已固定 Snapshot/Query，按月整理到临时映射数组，最后每证券字段写一个文件。`calendars/day.txt` 是完整开放日历，instrument 文件保存候选/成员区间，`.day.bin` 采用小端 float32，第一值为日历起始索引，后续保留原值与 NaN。manifest 固定查询、字段单位、身份映射、代码版本、文件摘要和容差。

下方实际导出，再由 Research adapter 调用 Qlib。重复导出验证后复用，改变 cutoff/字段须新目录；它不会隐式训练、填财务数据或计算策略特征。

固定 Snapshot、范围/字段、PIT/cutoffs 和映射复用同一持久目录；换模型或仅用已有字段改 Feature 公式不重导。Research 提供需求并消费，Data 负责格式导出。首次导出可作为一次性准备，不在每次策略研究重复支付成本。日历也先按 PIT 筛可见修订；未知日历会拒绝导出，不能取所有旧开市行的并集。


固定查询重复导出验证并命中原文件；测试检查文件 mtime 不变，不重读全量供应商。Research QlibView 已发布，Data 导出不依赖它。多字段、多报告期联合输入和持久复用已使用 Core asof 实现，真实小样本见 Researcher 教程末节。

In [37]:
sys.path.insert(0, str(WORKSPACE / 'axiom-research' / 'src'))
from axiom_data import verify_qlib_export
from axiom_research.qlib_adapter import QlibView
import numpy as np, shutil
qlib_spec = json.loads((REPO/'examples/qlib_stock_debug.spec.json').read_text())
qlib_queries = tuple(QuerySpec(**q) for q in qlib_spec['queries'])
qlib_members = QuerySpec(**qlib_spec['universe_query'])
qlib_target = TEMP/'qlib-native'
qlib_manifest = UNIFIED.export_qlib(snapshot=UNIFIED_SNAPSHOT,
    queries=qlib_queries,destination=qlib_target,universe_query=qlib_members,
    universe_name='csi1800')
print('实际临时导出:', qlib_target)
print('QlibView:', qlib_manifest['view_id'])
print('字段:', list(qlib_manifest['fields']))
display(pd.DataFrame(qlib_manifest['fields']).T)


实际临时导出: /var/folders/hq/xjtfsmm95nx1lpw034lk4cq00000gn/T/axiom-real-tutorial-0_wr_by8/qlib-native
QlibView: 1d1c55ff90848d5ea591ac6719461f5a8e143bf347b74f30003a8673e5b704ae
字段: ['open', 'high', 'low', 'close', 'volume_shares', 'amount_cny', 'factor', 'up_limit', 'down_limit']


,domain,field,source_dtype,unit,price_basis
open,market_daily,open,float64,CNY/share,unadjusted
high,market_daily,high,float64,CNY/share,unadjusted
low,market_daily,low,float64,CNY/share,unadjusted
close,market_daily,close,float64,CNY/share,unadjusted
volume_shares,market_daily,volume_shares,int64,shares,unadjusted
amount_cny,market_daily,amount_cny,float64,CNY,unadjusted
factor,adjustment_factors,factor,float64,dimensionless,unadjusted
up_limit,price_limits,up_limit,float64,CNY/share,unadjusted
down_limit,price_limits,down_limit,float64,CNY/share,unadjusted


In [38]:
print('实际目录:')
print('\n'.join(str(p.relative_to(qlib_target)) for p in sorted(qlib_target.rglob('*')) if p.is_file()))
print('固定日历前五行:', (qlib_target/'calendars/day.txt').read_text().splitlines()[:5])
print('成员区间:', (qlib_target/'instruments/csi1800.txt').read_text())
first_instrument = next(iter(qlib_manifest['instrument_map'].values()))
binary = np.fromfile(qlib_target/'features'/first_instrument.lower()/'close.day.bin',dtype='<f4')
print('第一个float32为日历起始下标:', binary[0], '；随后是实际 close:', binary[1:7])
print('浮点合同:', qlib_manifest['numeric_format'])
print('实测最大绝对rounding error:', qlib_manifest['max_absolute_float32_error'])


实际目录:
axiom-qlib.json
calendars/day.txt
features/cnstock.000001.sz.19910403/amount_cny.day.bin
features/cnstock.000001.sz.19910403/close.day.bin
features/cnstock.000001.sz.19910403/down_limit.day.bin
features/cnstock.000001.sz.19910403/factor.day.bin
features/cnstock.000001.sz.19910403/high.day.bin
features/cnstock.000001.sz.19910403/low.day.bin
features/cnstock.000001.sz.19910403/open.day.bin
features/cnstock.000001.sz.19910403/up_limit.day.bin
features/cnstock.000001.sz.19910403/volume_shares.day.bin
features/cnstock.600000.sh.19991110/amount_cny.day.bin
features/cnstock.600000.sh.19991110/close.day.bin
features/cnstock.600000.sh.19991110/down_limit.day.bin
features/cnstock.600000.sh.19991110/factor.day.bin
features/cnstock.600000.sh.19991110/high.day.bin
features/cnstock.600000.sh.19991110/low.day.bin
features/cnstock.600000.sh.19991110/open.day.bin
features/cnstock.600000.sh.19991110/up_limit.day.bin
features/cnstock.600000.sh.19991110/volume_shares.day.bin
instruments/all.txt
inst

In [39]:
verify_qlib_export(qlib_target,data=UNIFIED)
consumer = QlibView(qlib_target).activate()
qlib_native = consumer.read(fields=('close','volume_shares','factor'))
display(qlib_native.head(8))
members_native = consumer.read(fields=('close',),universe='csi1800')
print('实际 Research adapter → Qlib D.features:', len(qlib_native), '行')
print('命名成员集合:', len(members_native), '行')
print('P04 引用绑定:', consumer.reference['snapshot_id'], consumer.reference['view_id'])


$close  $volume_shares     $factor
instrument                 datetime                                      
CNSTOCK.000001.SZ.19910403 2026-06-01   10.99      95459568.0  134.579407
                           2026-06-02   11.08      88542840.0  134.579407
                           2026-06-03   10.99      82527168.0  134.579407
                           2026-06-04   10.82      86927080.0  134.579407
                           2026-06-05   10.98     100350336.0  134.579407
                           2026-06-08   11.03     109092648.0  134.579407
                           2026-06-09   11.13     114967232.0  134.579407
                           2026-06-10   11.32     154317632.0  134.579407

实际 Research adapter → Qlib D.features: 130 行
命名成员集合: 130 行
P04 引用绑定: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049 1d1c55ff90848d5ea591ac6719461f5a8e143bf347b74f30003a8673e5b704ae


In [40]:
file_times = {str(p.relative_to(qlib_target)):p.stat().st_mtime_ns
              for p in qlib_target.rglob('*') if p.is_file()}
reused = UNIFIED.export_qlib(snapshot=UNIFIED_SNAPSHOT,queries=qlib_queries,
    destination=qlib_target,universe_query=qlib_members,universe_name='csi1800')
assert reused['view_id'] == qlib_manifest['view_id']
assert file_times == {str(p.relative_to(qlib_target)):p.stat().st_mtime_ns
                     for p in qlib_target.rglob('*') if p.is_file()}
qlib_moved = TEMP/'qlib-relocated'; shutil.copytree(qlib_target,qlib_moved)
moved_view = QlibView(qlib_moved).activate()
pd.testing.assert_frame_equal(qlib_native,moved_view.read(fields=('close','volume_shares','factor')))
assert moved_view.reference == consumer.reference
verify_qlib_export(qlib_moved)
print('重复导出无写入；搬移后真实 Qlib 读取相同；源 current 不变。')


重复导出无写入；搬移后真实 Qlib 读取相同；源 current 不变。


## 13. ETF 独立来源与数据合同

基金有自己的稳定身份、日线单位、复权因子、限价和分红接口。七只 ETF 与沪深300基准覆盖真实三个月及预热，直接供 Research/Engine 读取；日线数据不能还原 09:35 的真实成交。执行改为当日开盘价的约定由 Engine 实施，Data 提供前日信号事实与当前开盘的单独 replay 请求。

In [41]:
# A second fixed, real Tushare Snapshot holds only the seven ETF data scope.
ETF_ROOT = Path(os.environ.get('AXIOM_ETF_DATA_ROOT',
    WORKSPACE / 'data/etf_rotation_daily_202606_202608'))
ETF_SNAPSHOT = os.environ.get('AXIOM_ETF_SNAPSHOT',
    's_e32f4511b69d4fce9accad609feb4005f371cc5cb833ca260d6ae563b75558d1')
ETF = Data(ETF_ROOT)
ETF_CURRENT_BEFORE = (ETF_ROOT/'current.json').read_bytes()
ETF_SCOPE = json.loads(Path(os.environ.get('AXIOM_ETF_SCOPE',
    REPO / 'delivery/etf_rotation_daily_202606_202608.scope.json')).read_text())
ETF_REPORT = json.loads(Path(os.environ.get('AXIOM_ETF_REPORT',
    REPO / 'delivery/etf_rotation_daily_202606_202608.report.json')).read_text())
assert ETF.resolve('current') == ETF_SNAPSHOT
assert ETF_REPORT['snapshot_id'] == ETF_SNAPSHOT
assert ETF_REPORT['verify']['verified'] and ETF_REPORT['audit']['status'] == 'passed'
etf_manifest = ETF.store.load_snapshot(ETF_SNAPSHOT)

In [42]:
etf_ids = tuple(ETF_SCOPE['identity_map'][symbol] for symbol in ETF_SCOPE['symbols'])
master = [r for p in etf_manifest['domains']['security_master']['partitions']
          for r in ETF.store.read_partition(p).to_pylist()]
assert {r['security_id'] for r in master} == set(etf_ids)
display(pd.DataFrame(master)[['source_code','security_id','name','asset_type','listing_date','list_status']])
calendar = [r for p in etf_manifest['domains']['trading_calendar']['partitions']
            for r in ETF.store.read_partition(p).to_pylist()]
open_sse = sorted(str(r['session']) for r in calendar if r['exchange'] == 'SSE'
                  and r['is_open'] and str(r['session']) <= ETF_SCOPE['end_session'])
print('ETF scope:', ETF_SCOPE['start_session'], '—', ETF_SCOPE['end_session'],
      '；预热从', ETF_SCOPE['warmup_start_session'], '开始；日线实际交易日:',
      len(open_sse), '；首尾:', open_sse[0], open_sse[-1])
assert sum(p['rows'] for p in etf_manifest['domains']['market_daily']['partitions']) == 7*114
assert sum(p['rows'] for p in etf_manifest['domains']['corporate_actions']['partitions']) == 44

,source_code,security_id,name,asset_type,listing_date,list_status
0,510880.SH,cn.etf.SSE.510880.20070118,华泰柏瑞上证红利ETF,exchange_traded_fund,2007-01-18,L
1,510300.SH,cn.etf.SSE.510300.20120528,华泰柏瑞沪深300ETF,exchange_traded_fund,2012-05-28,L
2,510500.SH,cn.etf.SSE.510500.20130315,南方中证500ETF,exchange_traded_fund,2013-03-15,L
3,511010.SH,cn.etf.SSE.511010.20130325,国泰上证5年期国债ETF,exchange_traded_fund,2013-03-25,L
4,513100.SH,cn.etf.SSE.513100.20130515,国泰纳斯达克100ETF(QDII),exchange_traded_fund,2013-05-15,L
5,518880.SH,cn.etf.SSE.518880.20130729,华安易富黄金ETF,exchange_traded_fund,2013-07-29,L
6,159915.SZ,cn.etf.SZSE.159915.20111209,易方达创业板ETF,exchange_traded_fund,2011-12-09,L


ETF scope: 2026-06-01 — 2026-08-31 ；预热从 2026-03-18 开始；日线实际交易日: 114 ；首尾: 2026-03-18 2026-08-31


In [43]:
def etf_read(domain, fields, day, cutoff, *, purpose='decision_facts', securities=etf_ids):
    return ETF.read(snapshot=ETF_SNAPSHOT, query=QuerySpec(domain, tuple(fields),
        tuple(securities), (day,), 'best_effort_vendor_v1', {day:cutoff}, purpose=purpose))

prior_day = '2026-08-28'
current_day = '2026-08-31'
prior_cutoff = prior_day+'T20:30:00+08:00'
prior_bar = etf_read('market_daily',
    ('open','high','low','close','volume_units','amount_cny'), prior_day, prior_cutoff)
prior_factor = etf_read('adjustment_factors', ('factor',), prior_day, prior_cutoff)
benchmark = etf_read('benchmark_daily', ('close',), prior_day, prior_cutoff,
    securities=('000300.SH',))
limits = etf_read('price_limits', ('up_limit','down_limit'), current_day,
    current_day+'T09:00:00+08:00')
# An opening price is a historical market-replay outcome. It is never a same-day
# input to a pre-open decision, and the rest of that day's bar is not queried here.

In [44]:
replay_open = etf_read('market_daily', ('open',), current_day,
    current_day+'T20:30:00+08:00', purpose='market_replay')
assert all(len(batch.frame) == 7 for batch in
           (prior_bar, prior_factor, limits, replay_open))
assert len(benchmark.frame) == 1
display(prior_bar.frame)
display(prior_factor.frame)
display(limits.frame)
display(benchmark.frame)
display(replay_open.frame)
print('单位:', {k:v['unit'] for k,v in prior_bar.field_meta.items()})
print('日线一条真实来源:', prior_bar.field_meta['close']['by_key'][0])
strict_old = ETF.read(snapshot=ETF_SNAPSHOT,
    query=QuerySpec('market_daily', ('close',), etf_ids, (prior_day,),
        'operational_pit_v1', {prior_day:prior_cutoff}))

,security_id,session,open,high,low,close,volume_units,amount_cny
0,cn.etf.SZSE.159915.20111209,2026-08-28,3.478,3.539,3.44,3.443,1485427127,5179583482.6
1,cn.etf.SSE.510300.20120528,2026-08-28,4.684,4.706,4.675,4.679,604693322,2835120646.0
2,cn.etf.SSE.510500.20130315,2026-08-28,7.96,8.004,7.913,7.923,267221620,2125117295.0
3,cn.etf.SSE.510880.20070118,2026-08-28,3.45,3.475,3.425,3.461,132670400,457855094.0
4,cn.etf.SSE.511010.20130325,2026-08-28,141.137,141.137,141.08,141.112,18388500,2594813442.0
5,cn.etf.SSE.513100.20130515,2026-08-28,2.244,2.247,2.229,2.235,108671134,243076831.0
6,cn.etf.SSE.518880.20130729,2026-08-28,9.437,9.475,9.398,9.475,473701359,4462695029.0


,security_id,session,factor
0,cn.etf.SZSE.159915.20111209,2026-08-28,1.0
1,cn.etf.SSE.510300.20120528,2026-08-28,1.2671
2,cn.etf.SSE.510500.20130315,2026-08-28,0.3401
3,cn.etf.SSE.510880.20070118,2026-08-28,1.8136
4,cn.etf.SSE.511010.20130325,2026-08-28,1.0375
5,cn.etf.SSE.513100.20130515,2026-08-28,5.0019
6,cn.etf.SSE.518880.20130729,2026-08-28,1.0


,security_id,session,up_limit,down_limit
0,cn.etf.SZSE.159915.20111209,2026-08-31,4.132,2.754
1,cn.etf.SSE.510300.20120528,2026-08-31,5.147,4.211
2,cn.etf.SSE.510500.20130315,2026-08-31,8.715,7.131
3,cn.etf.SSE.510880.20070118,2026-08-31,3.807,3.115
4,cn.etf.SSE.511010.20130325,2026-08-31,155.223,127.001
5,cn.etf.SSE.513100.20130515,2026-08-31,2.459,2.012
6,cn.etf.SSE.518880.20130729,2026-08-31,10.423,8.528


,security_id,session,close
0,000300.SH,2026-08-28,4609.1764


,security_id,session,open
0,cn.etf.SZSE.159915.20111209,2026-08-31,3.389
1,cn.etf.SSE.510300.20120528,2026-08-31,4.64
2,cn.etf.SSE.510500.20130315,2026-08-31,7.85
3,cn.etf.SSE.510880.20070118,2026-08-31,3.466
4,cn.etf.SSE.511010.20130325,2026-08-31,141.116
5,cn.etf.SSE.513100.20130515,2026-08-31,2.207
6,cn.etf.SSE.518880.20130729,2026-08-31,9.16


单位: {'open': 'CNY/fund unit', 'high': 'CNY/fund unit', 'low': 'CNY/fund unit', 'close': 'CNY/fund unit', 'volume_units': 'fund units', 'amount_cny': 'CNY'}
日线一条真实来源: {'security_id': 'cn.etf.SZSE.159915.20111209', 'session': '2026-08-28', 'revision_id': 'terminal:107290163259fbe5d5eff6c8c2a8b5d2df16e08f0b050fdb33333a56d575c949', 'revision_sequence': None, 'raw_batch_id': 'b_6a5de10d5b84462199a50c3970d8b133', 'usable_from': '2026-08-28T20:00:00+08:00', 'first_observed_at': '2026-10-03T06:01:34.876904+00:00', 'availability_basis': 'declared_vendor_assumption', 'evidence_ref': None, 'missing_reason': None}


In [45]:
assert strict_old.frame['close'].isna().all()

dividends = ETF.events(snapshot=ETF_SNAPSHOT,
    query=EventQuery('corporate_actions',
        ('cash_dividend_per_unit','ex_date','pay_date'), etf_ids,
        ETF_SCOPE['warmup_start_session'], ETF_SCOPE['end_session'],
        datetime.now(timezone.utc).isoformat(), 'operational_pit_v1',
        'announcement_date', {}))
display(dividends.frame)
print('分红原生键:', etf_manifest['domains']['corporate_actions']['contract']['logical_key'])
print('当前窗口内可见事件:', len(dividends.frame),
      '；全 Snapshot 来源事件行:',
      sum(p['rows'] for p in etf_manifest['domains']['corporate_actions']['partitions']))
print('旧日严格 cutoff 早于原 Raw 接收，不能回填日线可见性。')

,security_id,announcement_date,process_status,cash_dividend_per_unit,ex_date,pay_date
0,cn.etf.SSE.510500.20130315,2026-07-08,实施,0.149,2026-07-15,2026-07-20
1,cn.etf.SSE.511010.20130325,2026-03-20,实施,0.5715,2026-03-25,2026-03-30
2,cn.etf.SSE.511010.20130325,2026-06-22,实施,0.8144,2026-06-25,2026-06-30


分红原生键: ['security_id', 'announcement_date', 'process_status']
当前窗口内可见事件: 3 ；全 Snapshot 来源事件行: 44
旧日严格 cutoff 早于原 Raw 接收，不能回填日线可见性。


## 14. 附录：跨仓接口与职责

Data 不导入 Research、Engine 或 UI。消费者的薄 adapter 接受固定 Snapshot 与 Query，再按各自协议映射，不在查询时采集或修复。以下实际合同调用使用两个观察月末，验证引用与单位传递，完整账户回测另属 Engine/Trade。

| 调用链 | 真正传递的对象 | 本教程证明与边界 |
|---|---|---|
| Research → Data | `Data.read(snapshot, QuerySpec)` → `DataBatch` | 实际价格、字段来源、PIT；Research 定义 Feature/Label |
| Research adapter → Engine/Core | `FactBatch + ExecutionContext + FeaturePlan` | 真实月末事实执行 identity/lag/return；不是完整策略回测 |
| Engine/Runtime → Data | `DecisionBatchGate` 固定批次版本；decision 与 market_replay 分开 | 实际读取及引用记录；没有委托、成交、账户仿真 |
| Data / Research → UI | `records + field_meta + context` → P12 `ChartContext/ChartLayer` | 真实 JSON 图层；不是已经部署的 BFF/浏览器应用 |

四个 repo 同机函数调用即可。Core 不直接读 Parquet；UI 不自己重算事实，也不触发供应商请求。

**证据日期与当前源码：** 本附录保留此前薄接口实际输出，只证明当时调用；不当作新账户回测证据。2026-10-04 固定 ETF 消费链另由 Research 0.1.2、Engine 0.2.1、UI 0.1.0 验收并推送源码，严格对照和用户批准的日线近似分别记录于[当前交付](../docs/current-delivery.md)。完整模型/OOS、Runtime/Broker 和 UI 产品仍待验。本轮仅更新本 Notebook 的说明与入口并渲染保存输出，执行代码单元为 0。

In [46]:
sys.path.insert(0, str(WORKSPACE / 'axiom-research' / 'src'))
sys.path.insert(0, str(WORKSPACE / 'axiom-engine' / 'src'))
sys.path.insert(0, str(WORKSPACE / 'axiom-ui' / 'src'))
from axiom_engine.runtime import DecisionBatchGate
from axiom_research.data_adapter import adapt_decision_batch
from axiom_ui import chart_context, project_data_layer, unavailable_layer

# Runtime pins one concrete Snapshot and records both domain queries.
# The unified Snapshot includes two-security market facts and Tushare membership snapshots.
# This is a best-effort vendor-snapshot protocol demonstration, not a strict
# historical 2026-08 decision: the membership dependencies arrived later.
gate = DecisionBatchGate(UNIFIED)
gate.pin_session(MONTH_ENDS[-1], snapshot=MEMBER_SNAPSHOT)
clock = MONTH_ENDS[-1] + 'T20:30:00+08:00'
fact_query = query(sessions=MONTH_ENDS)
facts = gate.read_decision(decision_session=MONTH_ENDS[-1], query=fact_query, clock=clock)
vendor_member_query = replace(query(('is_member',), sessions=MONTH_ENDS,
    domain='universe_membership'), universe_id='csi1800')

In [47]:
refs = gate.read_decision(decision_session=MONTH_ENDS[-1], query=vendor_member_query, clock=clock)
assert refs.frame.is_member.tolist() == [True,True]
print('成员查询 policy:', refs.context['query']['pit_policy'],
      '；历史成员限制:', refs.context['limitations'])
recipe = 'sha256:' + hashlib.sha256(b'real month-end identity-lag-return tutorial v1').hexdigest()
adapted = adapt_decision_batch(facts, reference=refs, recipe_ref=recipe)
core_output = adapted.execute().to_dict()
display(pd.DataFrame(core_output['rows']))
print('Core输入单位:', adapted.plan.to_dict()['input_schema'][0]['unit'])
assert adapted.plan.to_dict()['input_schema'][0]['unit'] == 'CNY/share'
assert len(gate.references) == 2
# This return spans TWO OBSERVED MONTH-ENDS, not two consecutive trading days.
raw_close = facts.frame.close.tolist()
assert abs(core_output['rows'][1]['values'][2] - (raw_close[1]/raw_close[0]-1)) < 1e-12
view_for_ui = adapted.view_ref.to_dict()

成员查询 policy: best_effort_vendor_v1 ；历史成员限制: ['vendor release time is an assumption, not historical vintage evidence', 'Supplier index_weight was observed on dated snapshots only; no intramonth membership change is proven.', 'Best-effort uses an assumed 18:00 Asia/Shanghai dated release; strict policies use actual Raw receipt.']


,availability,cutoff,reasons,security_id,session,sources,valid,values
0,"[2026-07-31T12:00:00Z, None, 2026-07-31T12:00:...",2026-07-31T12:30:00Z,"[[], [INSUFFICIENT_HISTORY], [MISSING]]",cnstock.000001.SZ.19910403,2026-07-31,[[sha256:d730a0d0410cb729b0ad9ca3ec5ff2e38e66e...,"[True, False, False]","[11.63, None, None]"
1,"[2026-08-31T12:00:00Z, 2026-07-31T12:00:00Z, 2...",2026-08-31T12:30:00Z,"[[], [], []]",cnstock.000001.SZ.19910403,2026-08-31,[[sha256:bcf4678dadf562e45127ded94742996bdb540...,"[True, True, True]","[11.72, 11.63, 0.007738607050730817]"


Core输入单位: CNY/share


In [48]:
chart = chart_context(mode='run_replay', security_id=SYMBOL,
    session_refs={d: view_for_ui for d in MONTH_ENDS}, price_basis='unadjusted',
    adjustment_anchor=None, time_axis='decision_available')
layer = project_data_layer(facts, field='close', security_id=SYMBOL, role='data_fact', chart=chart)
display(pd.DataFrame(layer['points']))
assert [p['value'] for p in layer['points']] == raw_close
print('Data wire:', list(facts.to_response()))
print('UI协议:', layer['contract_version'], '固定版本:', layer['source_ref']['snapshot_id'])
print('无成交产物时 UI 明确 unavailable:', unavailable_layer(role='fill', requested_ref=None))
# Replay-side access is explicitly separate from decision-side facts.
market_query = replace(fact_query, purpose='market_replay')
market = gate.read_market(decision_session=MONTH_ENDS[-1], query=market_query, clock=clock)
assert market.context['query']['purpose'] == 'market_replay'

,security_id,session,value,provenance,missing_reason,source_available_time,observed_time
0,cnstock.000001.SZ.19910403,2026-07-31,11.63,"{'security_id': 'cnstock.000001.SZ.19910403', ...",None,2026-07-31T20:00:00+08:00,2026-09-28T15:06:01.854164+00:00
1,cnstock.000001.SZ.19910403,2026-08-31,11.72,"{'security_id': 'cnstock.000001.SZ.19910403', ...",None,2026-08-31T20:00:00+08:00,2026-09-28T15:06:14.947033+00:00


Data wire: ['records', 'field_meta', 'context']
UI协议: p12_chart_projection_v1 固定版本: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049
无成交产物时 UI 明确 unavailable: {'contract_version': 'p12_chart_projection_v1', 'role': 'fill', 'requested_ref': None, 'requested_stage': None, 'unit': None, 'status': 'LAYER_UNAVAILABLE', 'points': None, 'is_reconstructed': False, 'generated_at': '2026-10-03T15:05:13.908000+00:00'}


## 15. 附录：部署验证、容量与局限

验证包安装、采集恢复、源映射、查询、bundle 搬移与 Qlib 分别回答不同问题。一年固定1800市场样本用于吞吐和字节基线，两证券与ETF样本用于完整接口和事件映射。十二年耗时/容量仍是估计，要等真实历史并集准备后按范围调整。

In [49]:
from axiom_data.bulk_jobs import BulkJobPlan
plan_envelope = json.loads(ROOT.with_suffix('.plan.json').read_text())
# Plan envelope is a CLI artifact; use the public deserializer for its job.
frozen_job = BulkJobPlan.from_dict(plan_envelope['job'])
fresh_audit = audit_snapshot(DATA.store, snapshot_id=SNAPSHOT, plan=frozen_job)
display(pd.DataFrame([{'domain': k, 'source_mapping_rows_checked': v}
    for k, v in fresh_audit['source_mapping_checks'].items()]))
assert fresh_audit['source_mapping_checks'] == AUDIT['source_mapping_checks']
assert fresh_audit['missing_market_cells'] == fresh_audit['known_suspension_missing_cells'] == 158
print('本次重新离线核对:', fresh_audit['status'], '秒:', fresh_audit['elapsed_seconds'])


,domain,source_mapping_rows_checked
0,adjustment_factors,434911
1,benchmark_daily,242
2,market_daily,434464


本次重新离线核对: limited 秒: 11.779


In [50]:
sizes = pd.DataFrame([{'component': name, **measure_tree(ROOT/name)}
    for name in ('raw', 'canonical', 'snapshots', 'operations')])
display(sizes)
print('工作数据根:', measure_tree(ROOT))
print('当前搬移包:', measure_tree(Path(str(ROOT)+'-bundle')))
print('此前真实运行计时（非本次重新联网）:', RUN['timing'])
print('12×一年 active runtime:', round(RUN['timing']['bulk_active_process_seconds']*12/60, 1), '分钟')
print('12×一年目录 logical bytes:', round(measure_tree(ROOT)['bytes']*12/2**30, 2), 'GiB')


,component,files,bytes,MiB
0,raw,1791,252739100,241.03
1,canonical,70,44813913,42.74
2,snapshots,20,36544652,34.85
3,operations,33,1059911,1.01


工作数据根: {'files': 1916, 'bytes': 335157660, 'MiB': 319.63}
当前搬移包: {'files': 835, 'bytes': 252134085, 'MiB': 240.45}
此前真实运行计时（非本次重新联网）: {'bulk_active_process_seconds': 193.56, 'bulk_wall_including_fix_pause_seconds': 282.616, 'bulk_peak_rss_bytes': 327614464, 'offline_full_rebuild_seconds': 38.59, 'audit_seconds': 10.864, 'note': 'Two resumed real processes; no clean uninterrupted full-year or 12-year timing claim.'}
12×一年 active runtime: 38.7 分钟


12×一年目录 logical bytes: 3.75 GiB


In [51]:
full_sizes = pd.DataFrame([{'component':name, **measure_tree(FULL_ROOT/name)}
    for name in ('raw','canonical','snapshots','operations')])
display(full_sizes)
print('完整源小样本根合计:', measure_tree(FULL_ROOT))
print('这是两证券三个月及财务 warmup 的现场字节，不外推为十二年全市场实测。')


,component,files,bytes,MiB
0,raw,794,126359144,120.51
1,canonical,58,834293,0.80
2,snapshots,28,59176344,56.43
3,operations,81,887956,0.85


完整源小样本根合计: {'files': 963, 'bytes': 187257821, 'MiB': 178.58}
这是两证券三个月及财务 warmup 的现场字节，不外推为十二年全市场实测。


In [52]:
from collections import Counter
receipts = [json.loads(line) for line in (FULL_ROOT/'raw/fetches.jsonl').read_text().splitlines()]
payloads = [r['payload_sha256'] for r in receipts if r.get('payload_sha256')]
counts = Counter(payloads)
objects = {p.parent.name for p in (FULL_ROOT/'raw/objects').rglob('payload.bin')}
assert all(digest in objects for digest in counts)
print({'observations':len(receipts), 'payload_refs':len(payloads),
       'distinct_payloads':len(counts),
       'reused_payload_refs':sum(n-1 for n in counts.values()),
       'content_objects':len(objects)})
read_times = []
for label in ('first','repeat'):
    started = time.perf_counter()
    reread = full.events(snapshot=FULL_SNAPSHOT, query=income_query)
    read_times.append({'query':label,'rows':len(reread.frame),
        'seconds':round(time.perf_counter()-started,6)})
display(pd.DataFrame(read_times))
print('相同字节复用对象；重复观察保留接收记录。小样本读取计时仅描述此机器此查询。')


{'observations': 969, 'payload_refs': 969, 'distinct_payloads': 793, 'reused_payload_refs': 176, 'content_objects': 793}


,query,rows,seconds
0,first,7,0.028731
1,repeat,7,0.026691


相同字节复用对象；重复观察保留接收记录。小样本读取计时仅描述此机器此查询。


## 16. 附录：运行环境与样本证据

唯一编辑源是本仓两份 `.ipynb`，HTML 由它们生成。[Data 设计](../docs/design/02_axiom_data.md)规定目标，[总纲](../docs/design/01_axiom_overview.md)定义仓库边界；本教程展示当前实现及真实有限窗口，不把小样本当十二年验收。

初始化 helper 是 [tutorial_support.py](../examples/tutorial_support.py)，只读固定数据和本地验证文件、分配临时目录，不请求供应商。完整代码由附录入口重跑：安装 `requirements-notebooks.txt`，使用 `examples/real_tutorials.py` 执行已有 notebook 并生成 HTML。Qlib 章节另外需要 `requirements-qlib.lock`；跨仓附录需要同级 Engine、Research、UI 的薄 adapter。普通 Data 采集/Reader 只需基础包。

默认本地真实数据保存在 `../data/`，没有推送到 GitHub。路径可用 `AXIOM_WORKSPACE`、`AXIOM_TUTORIAL_DATA_ROOT`、`AXIOM_UNIFIED_DATA_ROOT` 等环境变量更改；本地 `delivery/` 保留采集与便携报告。源数据完整闭包经 bundle 搬移，并非仅克隆 repo 就会拥有这些样本。静态输出可直接阅读；全十二年准备/采集、跨操作系统搬移和整个 Engine 回测有各自的验证范围。

执行 notebook 会更新教学文件；若示例涉及离线构建，用 `AXIOM_TUTORIAL_DATA_PACKAGE` 指向留存并安装的当前 wheel 的 site-packages，使 builder 固定真实安装包。它不绕过源码/安装包的来源校验。

### 真实证据的范围

本轮交付包含Reader与Qlib两条读取路径。真实来源窗口验证映射与实际接口，独立语义反例验证自然样本没有出现的边界；最新测试数量、安装包与搬移证据以 `delivery/final-code-acceptance-20261003.json` 为准。

代码验收与十二年全量数据覆盖分别记录。生产事实接受Tushare供应商内容，外部核对只作warning；终态历史缺少的公开版本不被虚构为严格PIT证据。

| 范围 | 当前证据 | 尚未证明 |
|---|---|---|
| Raw / typed Canonical / Snapshot / Query、恢复及搬移 | 一年 1800 真实运行；两证券统一 Snapshot 的 14 启用域同机全域 Raw 重建与搬移 | 十二年实际运行；异操作系统搬移 |
| PIT、停牌与缺价、源映射 | Tushare 原响应、真实观察与反例夹具；生产上市/退市值不由外部资料改写 | 供应商历史字段的完整性；历史实际接收前的可见性 |
| 成员、Research/Core/Runtime/UI 薄接口 | Tushare 成员观测和价格进入真实薄接口，来源频率与延续规则在章节内明确 | 十二年逐日来源完整性、严格历史成员 PIT、完整模型/账户/Broker 与 UI 产品；本轮有界 ETF 链见[当前交付](../docs/current-delivery.md) |
| 财务 / 通用事件、TTM、撤销修订 | Tushare 两证券三个月完整源的报告期事件与 PIT 读取；金额 CNY 抽样核对；合成反例测试 | 十二年全池的供应商完整性；所有历史修订的公开时刻 |
| 七只 ETF 的日线数据 | 独立 Tushare Snapshot 的身份、114 交易日 OHLCV/因子/涨跌停、分红、日历和基准，真实 API 读取及8域搬移重建 | 十二年 ETF 范围、真实开盘流动性/撮合、完整策略平台；消费者短样本信号/账户另见[当前交付](../docs/current-delivery.md) |
| Qlib | 股票与ETF固定投影由实际Qlib 0.9.7读取；键、NaN及8352个数值单元与Reader等价，成员区间及换目录读取验证 | 全量导出测速；任意模型、processor或策略配方的等价性 |

验收证据与独立全量作业命令见仓库 `DELIVERY.md`、`delivery/final-code-acceptance-20261003.json`、`docs/design-conformance.md` 和 `docs/demo-acceptance.md`。供应商完整性、实际历史公开时刻、另一操作系统与消费者完整产品是证据或职责边界，不作为本轮尚待开发的Data代码。

In [53]:
assert DATA.resolve('current') == CURRENT_BEFORE
assert ((UNIFIED_ROOT/'current.json').read_bytes()
    if (UNIFIED_ROOT/'current.json').exists() else None) == UNIFIED_CURRENT_BEFORE
assert (ETF_ROOT/'current.json').read_bytes() == ETF_CURRENT_BEFORE
print('正式数据根的 current 未变化；教程无供应商调用。临时文件即将清理。')
TEMPDIR.cleanup()


正式数据根的 current 未变化；教程无供应商调用。临时文件即将清理。


## 17. 离线扩标：使用原 Raw 的显式新选择

下面是合成响应，专门验证“Raw 已保存全市场、Canonical 只收了 A”的边界。`domain_overrides.canonical_symbols` 可显式扩到 A+B，同时追加稳定身份。派生选择记入新 build context，原网络请求与 receipt 不改。所选域整体替换，需提供完整 Raw 闭包和目标选择；未选域保留，实际受影响月重建，不能把它称为自动只补新证券分区。0.3.5 的后续默认 rebuild 继承已生效的选择、合同和映射；新 operation 不需重复提交 override。下面再次重建并断言 A+B 仍在；普通 CLI 也有独立回归。Raw 没有 B 时仍缺失，需要另行明确补采。

多字段、多期的真实 Research 例子在 Researcher 教程末节；其 Core 计算和持久化复用已独立检查。字段从 Raw 扩展、时间追加与证券追加分别处理；不因正文或链接变化重采、重导出或全量重测。

In [2]:
from axiom_data import IngestBatch, UpdateRequest
from axiom_data.provider_local import CONTRACTS as SOURCE_CONTRACTS, profile_for

# Synthetic whole-market response: no supplier call and no formal-root writes.
offline=Data(TEMP/'offline-symbol-expansion')
ids={'000001.SZ':'sec-A','000002.SZ':'sec-B'}
raw_rows=[{'ts_code':code,'trade_date':'20200102','open':10,'high':11,'low':9,
    'close':10,'pre_close':9,'vol':1,'amount':1} for code in ids]
original=IngestBatch('market_daily',json.dumps(raw_rows).encode(),
    {'endpoint':'daily','params':{'trade_date':'20200102'},'canonical_symbols':['000001.SZ']},
    SOURCE_CONTRACTS['daily'],profile_for('daily',identity_map={'000001.SZ':'sec-A'}),
    datetime(2026,10,3,tzinfo=timezone.utc))
first=offline.update(base_snapshot=None,request=UpdateRequest((original,),
    'synthetic-original',{'synthetic':True}))
manifest=offline.store.load_snapshot(first.snapshot_id)
raw_ids=manifest['domains']['market_daily']['raw_batch_ids']
receipt_bytes=(offline.store.root/'raw/fetches.jsonl').read_bytes()
old_snapshot_bytes=(offline.store.root/'snapshots'/f'{first.snapshot_id}.json').read_bytes()
expanded=offline.rebuild(base_snapshot=first.snapshot_id,raw_batch_ids=raw_ids,
    domains=('market_daily',),operation_id='synthetic-offline-expansion',
    build_context={'synthetic':True,'reason':'explicitly expand saved Raw selection'},promote=False,
    domain_overrides={'market_daily':{'canonical_symbols':list(ids),
        'source_profile':profile_for('daily',identity_map=ids)}})
result=offline.read(snapshot=expanded.snapshot_id,query=QuerySpec('market_daily',('close',),
    tuple(ids.values()),('2020-01-02',),'operational_pit_v1',{'2020-01-02':'2026-10-04T00:00:00Z'}))
display(result.frame)
display(pd.DataFrame(result.field_meta['close']['by_key']))
assert result.frame.close.tolist()==[10,10]
# Ordinary default rebuild with a new operation: inherit effective A+B.
replayed=offline.rebuild(base_snapshot=expanded.snapshot_id,raw_batch_ids=raw_ids,
    domains=('market_daily',),operation_id='synthetic-default-rebuild',
    build_context={'synthetic':True},promote=False)
expanded_domain=offline.store.load_snapshot(expanded.snapshot_id)['domains']['market_daily']
replayed_domain=offline.store.load_snapshot(replayed.snapshot_id)['domains']['market_daily']
assert replayed_domain['partitions']==expanded_domain['partitions']
assert replayed_domain['source_profile']==expanded_domain['source_profile']
assert replayed_domain['build_context']['canonical_selection']=={'market_daily':list(ids)}
replayed_result=offline.read(snapshot=replayed.snapshot_id,query=QuerySpec('market_daily',('close',),
    tuple(ids.values()),('2020-01-02',),'operational_pit_v1',{'2020-01-02':'2026-10-04T00:00:00Z'}))
assert replayed_result.frame.close.tolist()==[10,10]
print('新 operation 默认 rebuild 继承 A+B，无需重复 override。')
assert (offline.store.root/'raw/fetches.jsonl').read_bytes()==receipt_bytes
assert (offline.store.root/'snapshots'/f'{first.snapshot_id}.json').read_bytes()==old_snapshot_bytes
assert offline.resolve('current')==first.snapshot_id
print('仅新建分支 Snapshot；原 Raw/receipt/current 不变，供应商请求 0。')


,security_id,session,close
0,sec-A,2020-01-02,10.0
1,sec-B,2020-01-02,10.0


,security_id,session,revision_id,revision_sequence,raw_batch_id,usable_from,first_observed_at,availability_basis,evidence_ref,missing_reason
0,sec-A,2020-01-02,terminal:b1aff9240533b40e4b5c54bf01e0e262b8a03...,None,b_514af82a04e84564b598aa40b27930e3,2026-10-03T00:00:00+00:00,2026-10-03T00:00:00+00:00,first_observed_at,None,None
1,sec-B,2020-01-02,terminal:8f628a202b500b043ef0c97996e28db6082d1...,None,b_514af82a04e84564b598aa40b27930e3,2026-10-03T00:00:00+00:00,2026-10-03T00:00:00+00:00,first_observed_at,None,None


新 operation 默认 rebuild 继承 A+B，无需重复 override。
仅新建分支 Snapshot；原 Raw/receipt/current 不变，供应商请求 0。
